# Dependencias

In [ ]:
import ctypes
ctypes.CDLL("/home/gcamilo/envs/wrf312/lib/libgomp.so.1", mode=ctypes.RTLD_GLOBAL)
import sys
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
import time
import glob
import random
import pandas as pd
import copy
import re
import wrf
import cartopy.feature as cfeature
import cartopy.crs as crs
import seaborn as sns

from torch.utils.data import Dataset, DataLoader
from torchinfo import summary
from tqdm.auto import tqdm
from datetime import datetime
from IPython.display import display
from matplotlib import animation
from matplotlib.colors import LinearSegmentedColormap
from skimage.metrics import structural_similarity as ssim
from wrf import getvar, get_cartopy, latlon_coords, to_np
from netCDF4 import Dataset as ncDataset
from scipy.ndimage import gaussian_filter1d

ruta_padre = os.path.abspath('..')
if ruta_padre not in sys.path:
    sys.path.append(ruta_padre)

CROP     = 1
# Rutas a las dos carpetas de salidas WRF (separadas por espacio en disco)
dir_2018 = os.path.expanduser("~/jupyter/salidasWRF2018")
dir_2019 = "../salidasWRF2019"
archivos = sorted(
    glob.glob(os.path.join(dir_2018, "wrfout_d01_*")) +
    glob.glob(os.path.join(dir_2019, "wrfout_d01_*"))
)
with ncDataset(archivos[0]) as ds0:
    ref = getvar(ds0, "LIGHTDENS")
    CART_PROJ = get_cartopy(ref)
    LATS, LONS = latlon_coords(ref)
LATS, LONS = to_np(LATS)[CROP:-CROP, CROP:-CROP], to_np(LONS)[CROP:-CROP, CROP:-CROP]

from modulos.tormentaDataset import TormentaDataset

# Chequeamos si hay GPU disponible para acelerar el entrenamiento
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Entrenando en: {device}")

In [ ]:
print("=== DIAGNÓSTICO DE PYTORCH Y CUDA ===")
print("Versión de PyTorch:", torch.__version__)
print("Versión de CUDA instalada en PyTorch:", torch.version.cuda)
print("¿CUDA disponible?:", torch.cuda.is_available())
print("Cantidad de GPUs detectadas:", torch.cuda.device_count())
print("=====================================\n")

if torch.cuda.is_available():
    try:
        device = torch.device("cuda")
        # Forzamos una operación matemática en la placa para validar el puente físico
        torch.zeros(1).to(device)
        print("✅ ¡Conexión exitosa a la GPU!")
        print(f"Propiedades del hardware asignado: {torch.cuda.get_device_name(0)}")
    except RuntimeError as e:
        print("❌ Error al interactuar con el hardware:", e)
else:
    print("⚠️ CUDA no está disponible en absoluto para este binario de PyTorch.")

In [ ]:
# Para REPRODUCIBILIDAD

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark     = False

# U-Net + FACL

Pérdida: **solamente FACL** (Yan et al., 2024, arXiv:2410.23159): selección aleatoria entre FAL y FCL con P(t) lineal de 1 a 0, sigmoide en la salida y cosine annealing.


In [ ]:
class UNet_WRF(nn.Module):
    """
    U-Net con profundidad y ancho configurables.

    Parámetros
    ----------
    in_channels : int, default 6
        Cantidad de canales de entrada (variables WRF).
    n_niveles : int, default 2
        Cantidad de niveles de encoder/decoder (pooling + upsampling).
        La arquitectura original tenía 2 niveles.
    n_filtros_base : int, default 32
        Cantidad de filtros en el primer nivel del encoder. Cada nivel
        siguiente duplica esta cantidad (32→64→128→...), igual que el
        bottleneck, que duplica una vez más el último nivel.
    kernel_sizes : list[int] o None
        Un kernel_size por nivel del encoder (y su espejo en el decoder).
        Si es None, usa kernel_size=3 en todos los niveles.
        Debe tener longitud == n_niveles.
    dropout_ps : list[float] o None
        Un dropout_p por nivel del encoder (y su espejo en el decoder),
        aplicado también en el bottleneck (usa el último valor de la lista).
        Si es None, no aplica dropout en ningún nivel (comportamiento original).
        Debe tener longitud == n_niveles.
    input_size : int, default 297
        Tamaño espacial (H=W) de la imagen de entrada, usado para calcular
        el padding necesario según la profundidad elegida.
    """
    def __init__(self, in_channels=6, n_niveles=2, n_filtros_base=32,
                 kernel_sizes=None, dropout_ps=None, input_size=297):
        super(UNet_WRF, self).__init__()

        if kernel_sizes is None:
            kernel_sizes = [3] * n_niveles
        assert len(kernel_sizes) == n_niveles, \
            f"kernel_sizes debe tener {n_niveles} elementos, tiene {len(kernel_sizes)}"

        if dropout_ps is None:
            dropout_ps = [0.0] * n_niveles
        assert len(dropout_ps) == n_niveles, \
            f"dropout_ps debe tener {n_niveles} elementos, tiene {len(dropout_ps)}"
        
        self.n_niveles = n_niveles
        self.dropout_ps = dropout_ps

        # --- Padding espacial dinámico según profundidad ---
        divisor = 2 ** n_niveles
        size_ajustado = ((input_size // divisor) + 1) * divisor
        pad_total = size_ajustado - input_size
        # Repartimos el padding: menos a la izquierda/arriba, más a la derecha/abajo
        self.pad_izq_arr = pad_total // 2
        self.pad_der_aba = pad_total - self.pad_izq_arr
        self.input_size = input_size

        # --- Canales por nivel
        canales = [n_filtros_base * (2 ** i) for i in range(n_niveles)]
        canal_bottleneck = n_filtros_base * (2 ** n_niveles)
        # Kernel del bottleneck: reusamos el del nivel más profundo
        k_bottleneck = kernel_sizes[-1]
        dropout_bottleneck = dropout_ps[-1]

        def bloque_conv(canal_in, canal_out, k, dropout_p):
            pad = k // 2
            capas = [
                nn.Conv2d(canal_in, canal_out, kernel_size=k, padding=pad),
                nn.ReLU(inplace=True),
                nn.Conv2d(canal_out, canal_out, kernel_size=k, padding=pad),
                nn.ReLU(inplace=True),
            ]
            if dropout_p > 0:
                capas.append(nn.Dropout2d(p=dropout_p))
            return nn.Sequential(*capas)
        
        # --- ENCODER (dinámico) ---
        self.enc_blocks = nn.ModuleList()
        self.pools      = nn.ModuleList()
        canal_in = in_channels
        for canal_out, k, p in zip(canales, kernel_sizes, dropout_ps):
            self.enc_blocks.append(bloque_conv(canal_in, canal_out, k, p))
            self.pools.append(nn.MaxPool2d(2, 2))
            canal_in = canal_out

        # --- BOTTLENECK ---
        self.bottleneck = bloque_conv(canal_in, canal_bottleneck, k_bottleneck, dropout_bottleneck)

        # --- DECODER (dinámico, en orden inverso al encoder) ---
        self.ups        = nn.ModuleList()
        self.dec_blocks = nn.ModuleList()
        canal_in = canal_bottleneck
        for canal_skip, k, p in zip(reversed(canales), reversed(kernel_sizes), reversed(dropout_ps)):
            self.ups.append(nn.Upsample(scale_factor=2, mode='bilinear', align_corners=True))
            self.dec_blocks.append(bloque_conv(canal_in + canal_skip, canal_skip, k, p))
            canal_in = canal_skip

        # --- SALIDA ---
        self.salida  = nn.Conv2d(canal_in, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()  # SOLO para FACL

    def forward(self, x):
        x_pad = F.pad(x, (self.pad_izq_arr, self.pad_der_aba,
                           self.pad_izq_arr, self.pad_der_aba))

        # --- Encoder ---
        skips = []
        h = x_pad
        for bloque, pool in zip(self.enc_blocks, self.pools):
            h = bloque(h)
            skips.append(h)       # guardamos para el skip connection
            h = pool(h)

        # --- Bottleneck ---
        h = self.bottleneck(h)

        # --- Decoder ---
        for up, bloque, skip in zip(self.ups, self.dec_blocks, reversed(skips)):
            h = up(h)
            h = torch.cat([h, skip], dim=1)
            h = bloque(h)

        out = self.salida(h)
        out = self.sigmoid(out)  # SOLO para FACL

        # Recortar el padding agregado al principio
        out = out[:, :, self.pad_izq_arr:self.pad_izq_arr + self.input_size,
                         self.pad_izq_arr:self.pad_izq_arr + self.input_size]
        return out

In [ ]:
# HP

modelo_unet = UNet_WRF(
    in_channels=3, 
    n_niveles=3, 
    n_filtros_base=64,
    kernel_sizes=[5, 3, 3], 
    dropout_ps=[0.0, 0.0, 0.0]
).to(device)

In [ ]:
print("\n\n=== RESUMEN DE LA U-NET ===")
summary(modelo_unet, input_size=(1, 3, 297, 297), 
        col_names=["input_size", "output_size", "num_params"])

# Entrenamiento

In [ ]:
# ==========================================
# FUNCION DE PÉRDIDA: FACL (Yan et al., NeurIPS 2024, arXiv:2410.23159)
# ==========================================
class FACL(nn.Module):
    """
    Fourier Amplitude and Correlation Loss, según el paper y su código oficial (https://github.com/argenycw/FACL).

        FAL(X, X^) = 1/(MN) * sum_pq ( |F_pq| - |F^_pq| )^2                          (Eq. 4)
        FCL(X, X^) = 1 - [ 1/2 * sum(F F^* + F^ F^*) ] / sqrt(sum|F|^2 * sum|F^|^2)  (Eq. 5)

        FACL(t) = FAL   si p > P(t)                                                  (Eq. 6)
                  FCL   si no
        con p ~ U[0,1] muestreado en cada paso de entrenamiento.

    P(t) decrece linealmente de 1 a 0: al principio se entrena 100% con FCL (estructura)
    y después cada vez más con FAL (nitidez). alpha es la fracción final de los pasos
    totales en la que P(t) = 0 (FAL puro). El paper recomienda alpha = 0.1 o 0.2.

    FCL GLOBAL: igual que el repo oficial y que RandomFourierLoss de rainpostproc, las sumas
    de la Eq. 5 se hacen sobre TODO el tensor (batch incluido), no imagen por imagen. Por
    Parseval, FCL = 1 - coseno entre los mapas aplanados, así que es el coseno del batch
    entero tomado como un único vector: las muestras más intensas pesan más.

    Requisitos del paper que se cumplen fuera de esta clase:
      - Sigmoide en la salida del modelo (ya está en UNet_WRF).
      - Scheduler cosine annealing (ya está: CosineAnnealingLR con T_max=EPOCHS).

    Parámetros
    ----------
    total_steps : int
        Pasos de optimización totales = EPOCHS * len(train_loader).
    alpha : float
        Fracción de pasos finales con P(t) = 0. Debe estar en [0, 1).
    eps : float
        Estabilidad numérica en el denominador de FCL. NO figura en la Eq. 5; solo entra en
        juego si el batch completo tiene target vacío (0/0).

    Validación (el paper no la define)
    ----------------------------------
    0.5*FAL + 0.5*FCL, calculada sobre TODO el set de validación de una vez, como en
    rainpostproc. Como FCL es global, promediar la pérdida por batches NO da lo mismo que
    calcularla sobre el set completo; por eso se acumulan las sumas batch a batch:

        criterion.val_reset()
        for x, y in val_loader:
            criterion.val_acumular(modelo(x), y)
        val_loss = criterion.val_resultado()

    forward(..., mode='val') sigue existiendo pero evalúa UN solo batch.
    """
    def __init__(self, total_steps, alpha=0.1, eps=1e-8):
        super().__init__()
        assert 0.0 <= alpha < 1.0, f"alpha es una fracción de los pasos totales, debe estar en [0, 1): {alpha}"
        self.total_steps = int(total_steps)
        self.alpha       = alpha
        self.eps         = eps
        self.step        = 0
        # Igual que el repo oficial: linspace(1, 0, T - int(alpha*T)) y después se queda en 0
        self.n_decay = max(int(self.total_steps - int(self.total_steps * alpha)), 2)
        self.val_reset()

    def prob_fcl(self, t=None):
        """P(t): probabilidad de elegir FCL en el paso t."""
        t = self.step if t is None else t
        return max(0.0, 1.0 - t / (self.n_decay - 1))

    @staticmethod
    def _fft(x):
        # DFT 2D ortonormalizada (Eq. 2), en float32 por estabilidad
        return torch.fft.fftn(x.float(), dim=[-2, -1], norm='ortho')

    def fal(self, fft_pred, fft_gt):
        # Eq. 4: media de (|F| - |F^|)^2 sobre todas las frecuencias (y el batch)
        return ((fft_gt.abs() - fft_pred.abs()) ** 2).mean()

    def fcl(self, fft_pred, fft_gt):
        # Eq. 5, con las sumas sobre todo el tensor (global).
        # Las dos normas van en raíces SEPARADAS: la de gt no lleva gradiente y vale 0 si el
        # target está vacío; sqrt(n_gt * n_pred) juntos daría gradiente NaN (inf * 0).
        cruzado    = 0.5 * (fft_gt * torch.conj(fft_pred) + fft_pred * torch.conj(fft_gt))
        num        = cruzado.real.sum()
        norma_pred = torch.sqrt((fft_pred.abs() ** 2).sum())
        norma_gt   = torch.sqrt((fft_gt.abs() ** 2).sum())
        return 1.0 - num / (norma_pred * norma_gt + self.eps)

    # ---- Validación sobre todo el set, acumulando sumas ----
    def val_reset(self):
        self._acc = {'num': 0.0, 'n_gt': 0.0, 'n_pred': 0.0, 'sq': 0.0, 'n': 0}

    @torch.no_grad()
    def val_acumular(self, pred, gt):
        fp, fg  = self._fft(pred), self._fft(gt)
        cruzado = 0.5 * (fg * torch.conj(fp) + fp * torch.conj(fg))
        self._acc['num']    += cruzado.real.sum(dtype=torch.float64).item()
        self._acc['n_gt']   += (fg.abs() ** 2).sum(dtype=torch.float64).item()
        self._acc['n_pred'] += (fp.abs() ** 2).sum(dtype=torch.float64).item()
        self._acc['sq']     += ((fg.abs() - fp.abs()) ** 2).sum(dtype=torch.float64).item()
        self._acc['n']      += fp.numel()

    def val_resultado(self):
        a = self._acc
        # float() a propósito: np.sqrt devuelve np.float64, y con PyTorch >= 2.6 torch.load
        # (weights_only=True) no puede leer escalares de numpy guardados en el checkpoint.
        fal = float(a['sq'] / a['n'])
        fcl = float(1.0 - a['num'] / (np.sqrt(a['n_pred']) * np.sqrt(a['n_gt']) + self.eps))
        self.val_fal, self.val_fcl = fal, fcl      # por si querés ver los dos términos por separado
        return float(0.5 * fal + 0.5 * fcl)

    def forward(self, pred, gt, mode='train'):
        fft_pred, fft_gt = self._fft(pred), self._fft(gt)

        if mode == 'val':      # un solo batch; para el set completo usar val_reset/val_acumular/val_resultado
            return 0.5 * self.fal(fft_pred, fft_gt) + 0.5 * self.fcl(fft_pred, fft_gt)

        p = np.random.rand()
        loss = self.fal(fft_pred, fft_gt) if p > self.prob_fcl() else self.fcl(fft_pred, fft_gt)
        self.step += 1
        return loss


In [ ]:
# ==========================================
# VERIFICACIÓN de FACL contra propiedades que demuestra el paper
# ==========================================
_c = FACL(total_steps=1000, alpha=0.1)
_x = torch.zeros(2, 1, 297, 297)
_x[:, :, 100:110, 150:160] = 0.8                      # "tormenta" sintética
_x[:, :, 200:203, 50:53]   = 0.5
_f = _c._fft

# FCL: invariante a escala global (Apéndice E) pero sensible a la posición
print("FCL(x, x)          =", _c.fcl(_f(_x), _f(_x)).item(), "  (esperado ~0)")
print("FCL(x, 0.3*x)      =", _c.fcl(_f(0.3 * _x), _f(_x)).item(), "  (esperado ~0)")
print("FCL(x, shift(x))   =", _c.fcl(_f(torch.roll(_x, (40, 30), (2, 3))), _f(_x)).item(), "  (esperado ~1)")
# FAL: invariante a traslación global (Apéndice D), sensible a la intensidad
print("FAL(x, shift(x))   =", _c.fal(_f(torch.roll(_x, (40, 30), (2, 3))), _f(_x)).item(), "  (esperado ~0)")
print("FAL(x, 0.3*x)      =", _c.fal(_f(0.3 * _x), _f(_x)).item(), "  (esperado > 0)")

# Batch con target vacío: ni el valor ni el gradiente pueden dar NaN
_p = torch.full((2, 1, 297, 297), 0.1, requires_grad=True)
_l = _c.fcl(_f(_p), _f(torch.zeros_like(_x))); _l.backward()
print("FCL(pred, vacío)   =", _l.item(), "| grad finito:", torch.isfinite(_p.grad).all().item(), "  (esperado 1.0 | True)")

# Validación acumulada por batches == calculada de una vez sobre todo el set
_g  = torch.Generator().manual_seed(0)
_y  = (torch.rand(5, 1, 297, 297, generator=_g) > 0.999).float() * torch.rand(5, 1, 297, 297, generator=_g)
_pr = torch.rand(5, 1, 297, 297, generator=_g)
_c.val_reset(); _c.val_acumular(_pr[:2], _y[:2]); _c.val_acumular(_pr[2:], _y[2:])
print(f"Val acumulada = {_c.val_resultado():.8f} | de una vez = {_c(_pr, _y, mode='val').item():.8f}  (esperado: iguales)")

print("tipo de val_resultado():", type(_c.val_resultado()).__name__, "  (esperado: float, no float64)")

# Schedule
print("P(FCL) en t=0, T/2, 0.9T, T:", [round(_c.prob_fcl(int(k * 1000)), 3) for k in (0, .5, .9, 1.)])
del _c, _x, _f, _p, _l, _g, _y, _pr


In [ ]:
# =====================================================================
# DATASETS
# =====================================================================

ruta_train = '../dataset/train'
ruta_val   = '../dataset/val'
ruta_stats = '../dataset/stats.npz'

print("Mapeando archivos y cargando estadísticas globales...")
train_dataset = TormentaDataset(split_dir=ruta_train, stats_path=ruta_stats)
val_dataset   = TormentaDataset(split_dir=ruta_val,   stats_path=ruta_stats)
print(f"-> Train: {len(train_dataset)} muestras | Val: {len(val_dataset)} muestras")
print(f"-> min_y: {train_dataset.min_y:.4f} | max_y: {train_dataset.max_y:.4f}")

vacias = 0
for _, y in DataLoader(train_dataset, batch_size=64, shuffle=False):
    vacias += (y.flatten(1).amax(dim=1) < 1e-6).sum().item()
print(f"-> Targets vacíos en train: {vacias} / {len(train_dataset)} ({100*vacias/len(train_dataset):.2f}%)")

In [ ]:
# =====================================================================
# HIPERPARÁMETROS
# =====================================================================

# --- Modelo ---
NOMBRE_MODELO = 'Ext Clip U-Net FACL (Optimized)'

# --- Optimización ---
LEARNING_RATE = 0.00012238028824652512
EPOCHS        = 140
PATIENCE      = 140
BATCH_SIZE    = 8

# --- FACL ---
ALPHA = 0.189085   # fracción final de pasos con FAL puro (el paper sugiere 0.1 - 0.2)

# ==========================================
# DATALOADERS
# ==========================================
generator = torch.Generator()
generator.manual_seed(SEED)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE,
                          shuffle=True, generator=generator,
                          num_workers=0, pin_memory=True)
val_loader   = DataLoader(val_dataset,   batch_size=BATCH_SIZE,
                          shuffle=False, num_workers=0, pin_memory=True)

print(f"-> Batches por época (Train): {len(train_loader)}")
print(f"-> Batches por época (Val):   {len(val_loader)}")

TOTAL_STEPS = EPOCHS * len(train_loader)

# --- Instanciación ---
modelo = UNet_WRF(
    in_channels=3, 
    n_niveles=3, 
    n_filtros_base=64,
    kernel_sizes=[5, 5, 3], 
    dropout_ps=[0.03, 0.08, 0.26]
).to(device)

criterion = FACL(total_steps=TOTAL_STEPS, alpha=ALPHA).to(device)

optimizer = optim.Adam(
    modelo.parameters(),
    lr=LEARNING_RATE
)

scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-7
)

# --- Historiales ---
historial_train_loss   = []
historial_val_loss     = []
historial_tiempo_epoch = []

print(f"-> Modelo:         {NOMBRE_MODELO}")
print(f"-> Corriendo en:   {device}")
print(f"-> Batch Size:     {BATCH_SIZE}")
print(f"-> Épocas totales: {EPOCHS}")
print(f"-> Patience (ES):  {PATIENCE} épocas")
print(f"-> Learning Rate:  {LEARNING_RATE:.6f}")
print(f"-> Función Loss:   FACL (alpha={ALPHA}, pasos totales={TOTAL_STEPS})")
print(f"-> P(FCL) al 30%/60%/90% del entrenamiento: "
      f"{criterion.prob_fcl(int(.3*TOTAL_STEPS)):.2f} / {criterion.prob_fcl(int(.6*TOTAL_STEPS)):.2f} / {criterion.prob_fcl(int(.9*TOTAL_STEPS)):.2f}")

In [ ]:
# ==========================================
# BUCLE DE ENTRENAMIENTO
# ==========================================
CARPETA_MODELOS = '../modelos'
mejor_val_loss = float('inf')
epocas_sin_mejora = 0
mejor_pesos = None
mejor_epoca = 0
historial_train_fal, historial_train_fcl = [], []
historial_val_fal,   historial_val_fcl   = [], []
historial_lr = []
fmt = lambda v: '--' if v != v else f"{v:.5f}"    # nan (ningún paso de ese término en la época) -> '--'

os.makedirs(CARPETA_MODELOS, exist_ok=True)
fecha = datetime.now().strftime("%Y%m%d_%H%M")
ruta_guardado = os.path.join(CARPETA_MODELOS, f"{fecha}_{NOMBRE_MODELO}.pth")

print(f"--- INICIANDO ENTRENAMIENTO (máx. {EPOCHS} épocas, patience={PATIENCE}) ---")
tiempo_inicio_total = time.time()
torch.cuda.empty_cache()

for epoch in range(EPOCHS):
    start_time = time.time()
    lr_epoca = optimizer.param_groups[0]['lr']
    
    # ==========================================
    # FASE DE ENTRENAMIENTO
    # ==========================================
    modelo.train()
    train_loss_acumulada = 0.0
    suma_fal, n_fal = 0.0, 0      # suma y cantidad de pasos que sortearon FAL
    suma_fcl, n_fcl = 0.0, 0      # ídem para FCL
    
    train_pbar = tqdm(train_loader, desc=f"Epoch {epoch+1:03d}/{EPOCHS} [Train]", leave=False)
    
    for batch_X, batch_y in train_pbar:
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        
        optimizer.zero_grad()                         
        predicciones = modelo(batch_X)         
        loss = criterion(predicciones, batch_y, mode='train')       
        loss.backward()    
        optimizer.step()                              
        
        loss_val = loss.item()
        train_loss_acumulada += loss_val * batch_X.size(0)
        if criterion.ultimo_termino == 'FAL':
            suma_fal += loss_val; n_fal += 1
        else:
            suma_fcl += loss_val; n_fcl += 1
        
        # Actualizamos el texto a la derecha de la barra para ver el loss en tiempo real
        train_pbar.set_postfix({'loss': f"{loss_val:.4f}", 'term': criterion.ultimo_termino,
                                'P(FCL)': f"{criterion.prob_fcl():.2f}"})
        
    epoch_train_loss = train_loss_acumulada / len(train_loader.dataset)
    historial_train_loss.append(epoch_train_loss)
    epoch_train_fal = suma_fal / n_fal if n_fal else float('nan')
    epoch_train_fcl = suma_fcl / n_fcl if n_fcl else float('nan')
    frac_fcl = n_fcl / (n_fal + n_fcl)
    historial_train_fal.append(epoch_train_fal); historial_train_fcl.append(epoch_train_fcl)
    
    # ==========================================
    # FASE DE VALIDACIÓN
    # ==========================================
    modelo.eval()
    criterion.val_reset()
    
    val_pbar = tqdm(val_loader, desc=f"Epoch {epoch+1:03d}/{EPOCHS} [Val]  ", leave=False)
    
    with torch.no_grad(): 
        for batch_X, batch_y in val_pbar:
            batch_X, batch_y = batch_X.to(device), batch_y.to(device)
            
            predicciones = modelo(batch_X)
            criterion.val_acumular(predicciones, batch_y)
            
            
    epoch_val_loss = criterion.val_resultado()
    historial_val_loss.append(epoch_val_loss)
    historial_val_fal.append(criterion.val_fal); historial_val_fcl.append(criterion.val_fcl)
    historial_lr.append(lr_epoca)
    tiempo_epoch = time.time() - start_time
    historial_tiempo_epoch.append(tiempo_epoch)

    # ==========================================
    # EARLY STOPPING
    # ==========================================
    if epoch_val_loss < mejor_val_loss:
        mejor_val_loss = epoch_val_loss
        epocas_sin_mejora = 0
        mejor_epoca = epoch + 1
        # Guardamos una copia en memoria de los mejores pesos
        mejor_pesos = copy.deepcopy(modelo.state_dict())
    else:
        epocas_sin_mejora += 1
    
    # ==========================================
    # GUARDADO
    # ==========================================
    torch.save({
        'model_state_dict': modelo.state_dict(),
        'best_model_state_dict': mejor_pesos,
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict':  scheduler.state_dict(),
        # V2.1 ONLY
        # 'criterion_state_dict': criterion.state_dict(),
        'historial_train_loss': historial_train_loss,
        'historial_val_loss': historial_val_loss,
        'historial_tiempo_epoch': historial_tiempo_epoch,
        'historial_train_fal': historial_train_fal,
        'historial_train_fcl': historial_train_fcl,
        'historial_val_fal': historial_val_fal,
        'historial_val_fcl': historial_val_fcl,
        'historial_lr': historial_lr,
        'mejor_epoca': mejor_epoca,
        'mejor_val_loss': mejor_val_loss,
        'epochs_entrenadas': epoch + 1,
        'lr': LEARNING_RATE,
        'patience': PATIENCE,
    }, ruta_guardado)

    # ==========================================
    # SCHEDULER STEP
    # ==========================================
    scheduler.step()
    lr_actual = optimizer.param_groups[0]['lr']
    
    # ==========================================
    # REPORTE DE LA ÉPOCA
    # ==========================================
    tiempo_epoch = time.time() - start_time
    print(f"Epoch {epoch+1:03d}/{EPOCHS} | Tiempo: {tiempo_epoch:.2f}s | "
          f"Train Loss: {epoch_train_loss:.6f} | Val Loss: {epoch_val_loss:.6f} | "
          f"Sin mejora: {epocas_sin_mejora}/{PATIENCE}")
    print(f"    LR: {lr_epoca:.2e} | pasos con FCL: {frac_fcl:.0%} | "
          f"Train FAL: {fmt(epoch_train_fal)}  FCL: {fmt(epoch_train_fcl)} | "
          f"Val FAL: {fmt(criterion.val_fal)}  FCL: {fmt(criterion.val_fcl)}")
    
    # ==========================================
    # CORTE POR EARLY STOPPING
    # ==========================================
    if epocas_sin_mejora >= PATIENCE:
        print(f"\nEarly stopping activado en la época {epoch+1}.")
        print(f"Mejor val_loss: {mejor_val_loss:.6f} (época {mejor_epoca})")
        break

# ==========================================
# RESTAURAR MEJORES PESOS
# ==========================================
modelo.load_state_dict(mejor_pesos)
print(f"\nPesos restaurados a la mejor época ({mejor_epoca}).")

# ==========================================
# REPORTE FINAL
# ==========================================
tiempo_fin_total = time.time()
segundos_totales = tiempo_fin_total - tiempo_inicio_total
horas_totales = segundos_totales / 3600
horas_enteras = int(horas_totales)
minutos_enteros = int((segundos_totales % 3600) / 60)

print("\n¡Entrenamiento finalizado exitosamente!")
print(f"Modelo final disponible en: {ruta_guardado}")
print(f"Tiempo total: {horas_totales:.2f} horas ({horas_enteras}h {minutos_enteros}m)")
print(f"Tiempo promedio por época: {np.mean(historial_tiempo_epoch):.1f}s")
print(f"Mejor época: {mejor_epoca} | Mejor val_loss: {mejor_val_loss:.6f}")

## Análisis de Importancia de Variables

In [ ]:
def cargar_set_en_memoria(loader):
    """Lee TODO el set una sola vez (acá se leen los .npz del disco). ~3.2 GB en RAM."""
    X, Y = [], []
    for x, y in tqdm(loader, desc='Cargando val en memoria'):
        X.append(x); Y.append(y)
    return torch.cat(X), torch.cat(Y)


def permutation_importance_global(modelo, X, Y, criterion, nombres_variables,
                                  n_repeats=100, batch_size=16, device='cuda', seed=None):
    """Permuta cada canal entre TODAS las muestras del set. Si se interrumpe, usa las repeticiones ya completas."""
    modelo.eval()
    g = torch.Generator()
    g.manual_seed(seed if seed is not None else int(torch.randint(0, 2**31, (1,))))
    N = X.shape[0]

    def evaluar(canal=None, perm=None):
        criterion.val_reset()
        with torch.no_grad():
            for i in range(0, N, batch_size):
                x = X[i:i + batch_size].clone()
                if canal is not None:
                    x[:, canal] = X[perm[i:i + batch_size], canal]      # valores de otras muestras
                criterion.val_acumular(modelo(x.to(device)), Y[i:i + batch_size].to(device))
        return criterion.val_resultado()

    baseline = evaluar()
    print(f"Pérdida Baseline (Val Loss sin permutar): {baseline:.6f}\n")

    filas = []                      # una fila por repetición COMPLETA: [delta_var0, delta_var1, ...]
    try:
        for rep in tqdm(range(n_repeats), desc='Repeticiones'):
            filas.append([evaluar(c, torch.randperm(N, generator=g)) - baseline
                          for c in range(len(nombres_variables))])
    except KeyboardInterrupt:
        print(f"\nInterrumpido: se usan las {len(filas)} repeticiones completas.")
    if not filas:
        raise RuntimeError("No se completó ninguna repetición.")

    d = np.array(filas)             # [repeticiones, variables]
    df = pd.DataFrame({'feature': nombres_variables, 'importance_mean': d.mean(0),
                       'importance_std': d.std(0), 'n_repeats': len(filas)}
                      ).sort_values('importance_mean', ascending=False)
    return df, baseline


def graficar_importancia(df_importances, nombre=None):
    nombre = nombre or NOMBRE_MODELO          # sin argumento, se comporta como antes
    plt.figure(figsize=(10, 6))
    plt.barh(df_importances['feature'], df_importances['importance_mean'], xerr=df_importances['importance_std'])
    plt.gca().invert_yaxis()
    plt.xlabel("Incremento medio en Val Loss (Permuted Loss - Baseline Loss)")
    plt.title("Permutation Feature Importance (U-Net FACL)")
    plt.grid(True, linestyle='--', alpha=0.5)
    plt.tight_layout()

    filename = f"../images/pfi_plot_{nombre}.png"
    plt.savefig(filename, dpi=300, bbox_inches='tight')
    print(f"Gráfico guardado exitosamente como {filename}")

    plt.show()

In [ ]:
nombres_wrf = ['MDBZ', 'HIELO', 'CTT']
TRIAL     = 39
N_REPEATS = 100

ruta = f'../modelos/etapa_B_Ext_Clip_U-Net_FACL/B_trial{TRIAL:04d}_Ext Clip U-Net FACL.pth'
checkpoint = torch.load(ruta, map_location=device, weights_only=False)   # checkpoint propio, de confianza
modelo.load_state_dict(checkpoint['best_model_state_dict'])              # mejor época
modelo.eval()
print(f"Trial {TRIAL} | mejor época {checkpoint['mejor_epoca']} | val guardado: {checkpoint['mejor_val_loss']:.6f}")

if 'X_val' not in globals():                  # se carga una sola vez; si re-corrés la celda, se reutiliza
    X_val, Y_val = cargar_set_en_memoria(val_loader)

df_imp, base_loss = permutation_importance_global(
    modelo, X_val, Y_val, criterion, nombres_wrf,
    n_repeats=N_REPEATS, batch_size=checkpoint['batch_size'], device=device, seed=SEED
)
print(f"Baseline recalculado: {base_loss:.6f}  (tiene que dar ~{checkpoint['mejor_val_loss']:.6f})")
print(df_imp)

ruta_csv = f"../estudios/pfi_global_trial{TRIAL:04d}_epoca{checkpoint['mejor_epoca']}.csv"
df_imp.to_csv(ruta_csv, index=False)          # para no tener que recalcular
graficar_importancia(df_imp, nombre=f"{NOMBRE_MODELO} etapaB trial{TRIAL:04d} epoca{checkpoint['mejor_epoca']} global")

# Optimización

In [ ]:
import subprocess, sys
result = subprocess.run([sys.executable, '-m', 'pip', 'install', 'optuna'], capture_output=True, text=True)
print(result.stdout[-2000:])
print(result.stderr[-2000:])

In [ ]:
import optuna

## Etapa A — Búsqueda de Arquitectura 

In [ ]:
# ==========================================
# ETAPA A — presupuesto FIJO por trial
# ==========================================
# El cronograma de FACL (P(t)) se escala con los pasos totales, así que todos los trials
# recorren la misma "película" (FCL -> FAL) y son comparables entre sí. Sin early stopping.
EPOCAS_TRIAL_A  = 50
BATCH_SIZE_A    = 16
LR_ETAPA_A      = 2.5e-5
ALPHA_A         = 0.1
WARMUP_PRUNER_A = int(0.6 * EPOCAS_TRIAL_A)       # el pruner recién actúa pasado el 60% del cronograma

def objective_arquitectura(trial):
    tiempo_inicio_trial = time.time()

    # ==========================================
    # HIPERPARÁMETROS DE ARQUITECTURA
    # ==========================================
    n_niveles      = trial.suggest_int('n_niveles', 3, 4)
    n_filtros_base = trial.suggest_categorical('n_filtros_base', [16, 32, 64])

    dropout_ps = [
        trial.suggest_float(f'dropout_nivel_{i}', 0.0, 0.3)
        for i in range(n_niveles)
    ]

    kernel_sizes = [
        trial.suggest_categorical(f'kernel_nivel_{i}', [3, 5, 7, 9])
        for i in range(n_niveles)
    ]

    # ==========================================
    # DATALOADERS
    # ==========================================
    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader_trial = DataLoader(train_dataset, batch_size=BATCH_SIZE_A,
                                     shuffle=True, generator=generator,
                                     num_workers=0, pin_memory=True)
    val_loader_trial   = DataLoader(val_dataset, batch_size=BATCH_SIZE_A,
                                     shuffle=False, num_workers=0, pin_memory=True)

    # ==========================================
    # MODELO, LOSS, OPTIMIZER Y SCHEDULER
    # ==========================================
    torch.manual_seed(SEED + trial.number)
    np.random.seed(SEED + trial.number)          # sorteo FAL/FCL reproducible por trial
    modelo_trial = UNet_WRF(
        in_channels=3,
        n_niveles=n_niveles, n_filtros_base=n_filtros_base,
        kernel_sizes=kernel_sizes, dropout_ps=dropout_ps
    ).to(device)
    n_parametros = sum(p.numel() for p in modelo_trial.parameters())

    criterion_trial = FACL(total_steps=EPOCAS_TRIAL_A * len(train_loader_trial),
                           alpha=ALPHA_A).to(device)

    optimizer_trial = optim.Adam(modelo_trial.parameters(), lr=LR_ETAPA_A)

    scheduler_trial = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_trial, T_max=EPOCAS_TRIAL_A, eta_min=1e-7
    )

    mejor_objetivo = float('inf')
    mejor_epoca    = 0
    val_fal_mejor  = val_fcl_mejor = None

    # ==========================================
    # BUCLE DE ENTRENAMIENTO DEL TRIAL
    # ==========================================
    try:
        for epoch in range(EPOCAS_TRIAL_A):
            modelo_trial.train()
            for batch_X, batch_y in train_loader_trial:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer_trial.zero_grad()
                pred = modelo_trial(batch_X)
                loss = criterion_trial(pred, batch_y, mode='train')
                loss.backward()
                optimizer_trial.step()

            # Validación sobre TODO el set (FCL global: se acumulan las sumas)
            modelo_trial.eval()
            criterion_trial.val_reset()
            with torch.no_grad():
                for batch_X, batch_y in val_loader_trial:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    criterion_trial.val_acumular(modelo_trial(batch_X), batch_y)
            epoch_val_loss = criterion_trial.val_resultado()

            # Detectar NaN/Inf y podar el trial en lugar de dejarlo completarse con basura
            if not np.isfinite(epoch_val_loss):
                print(f"[Trial {trial.number}] val_loss no finito ({epoch_val_loss}) — podando trial.")
                raise optuna.TrialPruned()

            scheduler_trial.step()

            if epoch_val_loss < mejor_objetivo:
                mejor_objetivo = epoch_val_loss
                mejor_epoca    = epoch + 1
                val_fal_mejor, val_fcl_mejor = criterion_trial.val_fal, criterion_trial.val_fcl

            trial.report(epoch_val_loss, epoch)
            if trial.should_prune():
                raise optuna.TrialPruned()

            print(f"[Trial {trial.number}] Epoch {epoch+1}/{EPOCAS_TRIAL_A} | val_loss={epoch_val_loss:.4f} "
                  f"(FAL={criterion_trial.val_fal:.5f} FCL={criterion_trial.val_fcl:.4f})")

        trial.set_user_attr('tiempo_trial_seg', time.time() - tiempo_inicio_trial)
        trial.set_user_attr('n_parametros', n_parametros)
        trial.set_user_attr('mejor_epoca', mejor_epoca)
        trial.set_user_attr('val_final', epoch_val_loss)
        trial.set_user_attr('val_fal_mejor', val_fal_mejor)
        trial.set_user_attr('val_fcl_mejor', val_fcl_mejor)
        return mejor_objetivo

    except optuna.TrialPruned:
        raise                                    

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f"[Trial {trial.number}] OOM — se descarta este trial.")
        raise optuna.TrialPruned()

    except Exception as e:
        torch.cuda.empty_cache()
        print(f"[Trial {trial.number}] Error inesperado: {type(e).__name__}: {e} — se descarta este trial.")
        raise optuna.TrialPruned()


In [ ]:
study_arq = optuna.create_study(
    study_name=f'busqueda_arquitectura_{NOMBRE_MODELO}',
    storage=f'sqlite:///../estudios/optuna_arquitectura_{NOMBRE_MODELO}.db',
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=WARMUP_PRUNER_A),
    load_if_exists=True
)

study_arq.optimize(objective_arquitectura, timeout=216*3600, catch=(Exception,))

In [ ]:
study_arq = optuna.load_study(
    study_name=f'busqueda_arquitectura_{NOMBRE_MODELO}',
    storage=f'sqlite:///../estudios/optuna_arquitectura_{NOMBRE_MODELO}.db'
)

print(f"Estudio cargado: {len(study_arq.trials)} trials en total")

In [ ]:
# =====================================================================
# RESUMEN DE LA ETAPA A — BÚSQUEDA DE ARQUITECTURA
# =====================================================================

trials_completados = [t for t in study_arq.trials if t.state == optuna.trial.TrialState.COMPLETE]
trials_podados      = [t for t in study_arq.trials if t.state == optuna.trial.TrialState.PRUNED]
trials_fallidos     = [t for t in study_arq.trials if t.state == optuna.trial.TrialState.FAIL]

print("="*60)
print("RESUMEN ETAPA A — BÚSQUEDA DE ARQUITECTURA")
print("="*60)
print(f"Total de trials registrados:   {len(study_arq.trials)}")
print(f"  -> Completados exitosamente: {len(trials_completados)}")
print(f"  -> Podados (pruned):         {len(trials_podados)}")
print(f"  -> Fallidos:                 {len(trials_fallidos)}")

# --- Detectar cuántos completados terminaron en NaN ---
trials_nan = [t for t in trials_completados if t.value is not None and np.isnan(t.value)]
print(f"  -> Completados con valor NaN: {len(trials_nan)}")

print("\n" + "-"*60)
print("MEJOR TRIAL ENCONTRADO")
print("-"*60)
if len(trials_completados) > 0:
    print(f"Trial número: {study_arq.best_trial.number}")
    print(f"Val Loss:     {study_arq.best_trial.value:.6f}")
    print(f"\nHiperparámetros de arquitectura:")
    for k, v in study_arq.best_trial.params.items():
        print(f"  {k}: {v}")
    # --- Obtención segura de atributos ---
    n_params = study_arq.best_trial.user_attrs.get('n_parametros', 'N/D')
    n_params_fmt = f"{n_params:,}" if isinstance(n_params, (int, float)) else n_params
    
    tiempo_seg = study_arq.best_trial.user_attrs.get('tiempo_trial_seg', 'N/D')
    tiempo_fmt = f"{tiempo_seg/60:.2f} minutos" if isinstance(tiempo_seg, (int, float)) else tiempo_seg
    
    print(f"\nCantidad de parámetros del modelo: {n_params_fmt}")
    print(f"Tiempo del trial: {tiempo_fmt}")
else:
    print("⚠️ Ningún trial se completó con éxito.")

print("\n" + "-"*60)
print("TOP 5 MEJORES TRIALS (por Val Loss)")
print("-"*60)
trials_df = study_arq.trials_dataframe()
trials_df_validos = trials_df[trials_df['state'] == 'COMPLETE'].sort_values('value')
cols_mostrar = ['number', 'value'] + [c for c in trials_df_validos.columns if c.startswith('params_')]
display(trials_df_validos[cols_mostrar].head(5))

from optuna.importance import FanovaImportanceEvaluator
evaluador_fijo = FanovaImportanceEvaluator(seed=SEED)

print("\n" + "-"*60)
print("IMPORTANCIA DE HIPERPARÁMETROS")
print("-"*60)
try:
    importancias = optuna.importance.get_param_importances(study_arq, evaluator=evaluador_fijo)
    for k, v in importancias.items():
        print(f"  {k}: {v:.4f}")
except Exception as e:
    print(f"No se pudo calcular importancia: {e}")

print("\n" + "-"*60)
print("TIEMPO TOTAL DE LA ETAPA")
print("-"*60)
tiempos = [t.user_attrs.get('tiempo_trial_seg', 0) for t in trials_completados]
if tiempos:
    print(f"Tiempo total acumulado en trials completados: {sum(tiempos)/3600:.2f} horas")
    print(f"Tiempo promedio por trial completado:         {np.mean(tiempos)/60:.1f} minutos")

# --- Gráfico de importancia ---
import optuna.visualization.matplotlib as vis_mpl

vis_mpl.plot_param_importances(study_arq, evaluator=evaluador_fijo)
plt.tight_layout()
plt.show()

# =====================================================================
# DETALLE DE LOS TOP 5: época del mínimo, FAL/FCL y curvas
# =====================================================================
top5 = sorted(trials_completados, key=lambda t: t.value)[:5]
fmt = lambda x, f: f"{x:{f}}" if isinstance(x, (int, float)) else "N/D"

print("\n" + "-"*60)
print("TOP 5: ÉPOCA DEL MÍNIMO Y COMPONENTES FAL / FCL")
print("-"*60)
print(f"{'trial':>6} {'val':>9} {'mejor_ep':>9} {'val_final':>10} {'FAL':>10} {'FCL':>8}")
for t in top5:
    ua = t.user_attrs
    print(f"{t.number:>6} {t.value:>9.5f} {str(ua.get('mejor_epoca', 'N/D')):>9} "
          f"{fmt(ua.get('val_final'), '.5f'):>10} "
          f"{fmt(ua.get('val_fal_mejor'), '.5f'):>10} "
          f"{fmt(ua.get('val_fcl_mejor'), '.4f'):>8}")

# --- Curvas de val_loss por época (completas y zoom al final) ---
fig, axs = plt.subplots(1, 2, figsize=(14, 4.5))
for t in top5:
    ep = sorted(t.intermediate_values)
    axs[0].plot([e + 1 for e in ep], [t.intermediate_values[e] for e in ep],
                label=f"#{t.number} ({t.value:.5f})")
    fin = [(e + 1, t.intermediate_values[e]) for e in ep if e + 1 >= 35]
    axs[1].plot(*zip(*fin), marker="o", ms=3, label=f"#{t.number}")
axs[0].set(title="Val loss por época (top 5)", xlabel="época", ylabel="val_loss")
axs[1].set(title="Zoom: épocas 35-50", xlabel="época", ylabel="val_loss")
for a in axs:
    a.grid(alpha=0.3)
    a.legend()
plt.tight_layout()
plt.show()

In [ ]:
import optuna.visualization.matplotlib as vis_mpl

df = study_arq.trials_dataframe()
d3 = df[(df.state == "COMPLETE") & (df.params_n_niveles == 3)].copy()

tab = (d3.groupby(["params_n_filtros_base", "params_kernel_nivel_1"]).value
         .agg(["count", "min", "median"]).round(5))
print(tab)

grupos = sorted(d3.groupby(["params_n_filtros_base", "params_kernel_nivel_1"]).groups)
datos = [d3[(d3.params_n_filtros_base == f) & (d3.params_kernel_nivel_1 == k)].value.values
         for f, k in grupos]

plt.figure(figsize=(11, 4.5))
plt.boxplot(datos, showfliers=False)
for i, d in enumerate(datos, 1):
    plt.scatter(np.random.normal(i, 0.04, len(d)), d, s=12, alpha=0.5)
plt.xticks(range(1, len(grupos) + 1), [f"f={f}\nk1={k}" for f, k in grupos])
plt.ylabel("val_loss (mejor)")
plt.title("3 niveles: val_loss por n_filtros_base y kernel_nivel_1")
plt.grid(alpha=0.3)
plt.tight_layout()
plt.show()

vis_mpl.plot_slice(study_arq, params=["n_filtros_base", "kernel_nivel_1", "kernel_nivel_2"])
plt.tight_layout()
plt.show()

In [ ]:
d = df[df.state == "COMPLETE"].copy()
d["horas"] = d.user_attrs_tiempo_trial_seg / 3600
d["params_M"] = d.user_attrs_n_parametros / 1e6

def pareto(dd, x):
    mejor, idx = np.inf, []
    for i, r in dd.sort_values(x).iterrows():
        if r.value < mejor:
            mejor = r.value
            idx.append(i)
    return dd.loc[idx]

fig, axs = plt.subplots(1, 2, figsize=(14, 4.8))
for ax, x, xl in zip(axs, ["params_M", "horas"], ["Parámetros (M)", "Tiempo del trial (h)"]):
    sc = ax.scatter(d[x], d.value, c=d.params_n_filtros_base, cmap="viridis", s=28)
    p = pareto(d, x)
    ax.step(p[x], p.value, where="post", color="red", lw=1.5, label="frente de Pareto")
    ax.set_xlabel(xl); ax.set_ylabel("val_loss")
    ax.set_ylim(d.value.min() * 0.99, d.value.quantile(0.95))
    ax.grid(alpha=0.3); ax.legend()
fig.colorbar(sc, ax=axs, label="n_filtros_base")
plt.show()

print("Frente de Pareto (valor vs parámetros):")
print(pareto(d, "params_M")[["number", "value", "params_M", "horas",
                              "params_n_niveles", "params_n_filtros_base"]].round(4))

## Etapa B — Optimización de Hiperparámetros de Entrenamiento

In [ ]:
ARQ_FINAL = dict(
    n_niveles=3,
    n_filtros_base=32,
    kernel_sizes=[9, 9, 3],
    dropout_ps=[0.08473501730018196, 0.07849710178981281, 0.32307659365394653],
)

In [ ]:
def objective_entrenamiento(trial):
    torch.cuda.empty_cache()
    tiempo_inicio_trial = time.time()

    # ==========================================
    # HIPERPARÁMETROS DE ENTRENAMIENTO A OPTIMIZAR
    # ==========================================
    lr     = trial.suggest_float('lr', 5e-5, 1e-3, log=True)
    alpha  = trial.suggest_float('alpha', 0.05, 0.30)          # el paper sugiere 0.1 - 0.2
    epocas = trial.suggest_int('epocas', 20, 200, step=10)     # el cronograma de FACL se ajusta a este total
    batch_size = trial.suggest_categorical('batch_size', [8, 16, 32])   # con FCL global también cambia la loss

    # ==========================================
    # CONFIGURACIÓN FIJA DE LA ETAPA B
    # ==========================================
    BATCH_VAL_B = 16       # la validación acumula sobre todo el set: su resultado no depende de este valor
    # Sin early stopping ni pruner: las épocas ya son un hiperparámetro de la búsqueda.

    # ==========================================
    # DATALOADERS
    # ==========================================
    generator = torch.Generator()
    generator.manual_seed(SEED)

    train_loader_trial = DataLoader(train_dataset, batch_size=batch_size,
                                     shuffle=True, generator=generator,
                                     num_workers=0, pin_memory=True)
    val_loader_trial   = DataLoader(val_dataset, batch_size=BATCH_VAL_B,
                                     shuffle=False, num_workers=0, pin_memory=True)

    # ==========================================
    # MODELO — arquitectura FIJA (ganadora de la Etapa A)
    # ==========================================
    torch.manual_seed(SEED + trial.number)
    np.random.seed(SEED + trial.number)          # sorteo FAL/FCL reproducible por trial
    modelo_trial = UNet_WRF(
        in_channels=3,
        n_niveles=ARQ_FINAL['n_niveles'],
        n_filtros_base=ARQ_FINAL['n_filtros_base'],
        kernel_sizes=ARQ_FINAL['kernel_sizes'],
        dropout_ps=ARQ_FINAL['dropout_ps'],
    ).to(device)

    criterion_trial = FACL(total_steps=epocas * len(train_loader_trial), alpha=alpha).to(device)

    optimizer_trial = optim.Adam(modelo_trial.parameters(), lr=lr)

    scheduler_trial = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer_trial, T_max=epocas, eta_min=1e-7
    )

    mejor_val_loss = float('inf')
    mejor_epoca    = 0
    val_fal_mejor  = val_fcl_mejor = None

    # ==========================================
    # BUCLE DE ENTRENAMIENTO DEL TRIAL
    # ==========================================
    try:
        for epoch in range(epocas):
            modelo_trial.train()
            for batch_X, batch_y in train_loader_trial:
                batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                optimizer_trial.zero_grad()
                pred = modelo_trial(batch_X)
                loss = criterion_trial(pred, batch_y, mode='train')
                loss.backward()
                optimizer_trial.step()

            # Validación sobre TODO el set (FCL global: se acumulan las sumas)
            modelo_trial.eval()
            criterion_trial.val_reset()
            with torch.no_grad():
                for batch_X, batch_y in val_loader_trial:
                    batch_X, batch_y = batch_X.to(device), batch_y.to(device)
                    criterion_trial.val_acumular(modelo_trial(batch_X), batch_y)
            epoch_val_loss = criterion_trial.val_resultado()

            # --- Chequeo de NaN/Inf ---
            if not np.isfinite(epoch_val_loss):
                print(f"[Trial {trial.number}] val_loss no finito ({epoch_val_loss}) — podando trial.")
                raise optuna.TrialPruned()

            scheduler_trial.step()

            if epoch_val_loss < mejor_val_loss:
                mejor_val_loss = epoch_val_loss
                mejor_epoca    = epoch + 1
                val_fal_mejor, val_fcl_mejor = criterion_trial.val_fal, criterion_trial.val_fcl

            print(f"[Trial {trial.number}] Epoch {epoch+1}/{epocas} | val_loss={epoch_val_loss:.4f} "
                  f"(FAL={criterion_trial.val_fal:.5f} FCL={criterion_trial.val_fcl:.4f})")

        trial.set_user_attr('tiempo_trial_seg', time.time() - tiempo_inicio_trial)
        trial.set_user_attr('mejor_epoca', mejor_epoca)
        trial.set_user_attr('val_final', epoch_val_loss)
        trial.set_user_attr('val_fal_mejor', val_fal_mejor)
        trial.set_user_attr('val_fcl_mejor', val_fcl_mejor)
        return mejor_val_loss

    except optuna.TrialPruned:
        raise                                    # no tratarlo como "error inesperado"

    except torch.cuda.OutOfMemoryError:
        torch.cuda.empty_cache()
        print(f"[Trial {trial.number}] OOM — se descarta este trial.")
        raise optuna.TrialPruned()

    except Exception as e:
        torch.cuda.empty_cache()
        print(f"[Trial {trial.number}] Error inesperado: {type(e).__name__}: {e} — se descarta este trial.")
        raise optuna.TrialPruned()


In [ ]:
import os
import shutil

# 1. Definimos las rutas
ruta_red = f'../estudios/optuna_entrenamiento_{NOMBRE_MODELO}.db'
ruta_local = f'/tmp/optuna_entrenamiento_{NOMBRE_MODELO}.db'

# 2. Si ya tenías un estudio previo en tu carpeta de red, lo traemos al disco local para continuarlo
if os.path.exists(ruta_red):
    print("Copiando base de datos de la red al disco local temporal...")
    shutil.copy2(ruta_red, ruta_local)

# 3. Configuramos Optuna para que use la ruta LOCAL
storage_local = f'sqlite:///{ruta_local}'

study_entrenamiento = optuna.create_study(
    study_name=f'optimizacion_entrenamiento_{NOMBRE_MODELO}',
    storage=storage_local,
    direction='minimize',
    sampler=optuna.samplers.TPESampler(seed=SEED),
    pruner=optuna.pruners.MedianPruner(n_warmup_steps=25),
    load_if_exists=True
)

print("Iniciando optimización local...")

# 4. Corremos la optimización
try:
    study_entrenamiento.optimize(objective_entrenamiento, timeout=161*3600, catch=(Exception,))
finally:
    # 5. Pase lo que pase (incluso si lo frenas a mano o hay un error), 
    # guardamos los datos de vuelta en tu carpeta de red.
    print("Guardando resultados en la unidad de red...")
    shutil.copy2(ruta_local, ruta_red)
    print("¡Base de datos respaldada con éxito!")

In [ ]:
study_entrenamiento = optuna.load_study(
    study_name=f'optimizacion_entrenamiento_{NOMBRE_MODELO}',
    storage=f'sqlite:///../estudios/optuna_entrenamiento_{NOMBRE_MODELO}.db'
)

print(f"Estudio cargado: {len(study_entrenamiento.trials)} trials en total")

In [ ]:
# =====================================================================
# RESUMEN DE LA ETAPA B — OPTIMIZACIÓN DE HIPERPARÁMETROS DE ENTRENAMIENTO
# =====================================================================

trials_completados = [t for t in study_entrenamiento.trials if t.state == optuna.trial.TrialState.COMPLETE]
trials_podados      = [t for t in study_entrenamiento.trials if t.state == optuna.trial.TrialState.PRUNED]
trials_fallidos     = [t for t in study_entrenamiento.trials if t.state == optuna.trial.TrialState.FAIL]
trials_interrumpidos = [t for t in study_entrenamiento.trials if t.state == optuna.trial.TrialState.RUNNING]

print("="*60)
print("RESUMEN ETAPA B — OPTIMIZACIÓN DE ENTRENAMIENTO")
print("="*60)
print(f"Total de trials registrados:   {len(study_entrenamiento.trials)}")
print(f"  -> Completados exitosamente: {len(trials_completados)}")
print(f"  -> Podados (pruned):         {len(trials_podados)}")
print(f"  -> Fallidos:                 {len(trials_fallidos)}")
print(f"  -> Interrumpidos (RUNNING):  {len(trials_interrumpidos)}")

trials_nan = [t for t in trials_completados if t.value is not None and np.isnan(t.value)]
print(f"  -> Completados con valor NaN: {len(trials_nan)}")

print("\n" + "-"*60)
print("MEJOR TRIAL ENCONTRADO")
print("-"*60)
if len(trials_completados) > 0:
    print(f"Trial número: {study_entrenamiento.best_trial.number}")
    print(f"Val Loss:     {study_entrenamiento.best_trial.value:.6f}")
    print(f"\nHiperparámetros de entrenamiento:")
    for k, v in study_entrenamiento.best_trial.params.items():
        print(f"  {k}: {v}")
    print(f"\nTiempo del trial: {study_entrenamiento.best_trial.user_attrs.get('tiempo_trial_seg', 'N/D')}")
else:
    print("⚠️ Ningún trial se completó con éxito.")

print("\n" + "-"*60)
print("TOP 5 MEJORES TRIALS (por Val Loss)")
print("-"*60)
trials_df = study_entrenamiento.trials_dataframe()
trials_df_validos = trials_df[trials_df['state'] == 'COMPLETE'].sort_values('value')
cols_mostrar = ['number', 'value'] + [c for c in trials_df_validos.columns if c.startswith('params_')]
display(trials_df_validos[cols_mostrar].head(5))

from optuna.importance import FanovaImportanceEvaluator
evaluador_fijo = FanovaImportanceEvaluator(seed=SEED)

print("\n" + "-"*60)
print("IMPORTANCIA DE HIPERPARÁMETROS")
print("-"*60)
try:
    importancias_reproducibles = optuna.importance.get_param_importances(
        study_entrenamiento, evaluator=evaluador_fijo
    )
    for k, v in importancias_reproducibles.items():
        print(f"  {k}: {v:.4f}")
except Exception as e:
    print(f"No se pudo calcular importancia (default): {e}")

print("\n" + "-"*60)
print("TIEMPO TOTAL DE LA ETAPA")
print("-"*60)
tiempos = [t.user_attrs.get('tiempo_trial_seg', 0) for t in trials_completados]
if tiempos:
    print(f"Tiempo total acumulado en trials completados: {sum(tiempos)/3600:.2f} horas")
    print(f"Tiempo promedio por trial completado:         {np.mean(tiempos)/60:.1f} minutos")

# --- Gráfico de importancia ---
import optuna.visualization.matplotlib as vis_mpl

optuna.logging.set_verbosity(optuna.logging.WARNING)   # saca el aviso de "new study in memory"
vis_mpl.plot_param_importances(s2, evaluator=FanovaImportanceEvaluator(seed=SEED))
plt.tight_layout(); plt.show()

# V2.1 ONLY

# --- Coeficientes de las funciones de pérdida ---

# print("\n" + "-"*60)
# print("PESOS DE UNCERTAINTY WEIGHTING (log_var finales)")
# print("-"*60)

# if len(trials_completados) > 0:
    # print("Mejor trial:")
    # print(f"  log_var_wmse: {study_arq.best_trial.user_attrs.get('log_var_wmse_final', 'N/D'):.4f}")
    # print(f"  log_var_fal:  {study_arq.best_trial.user_attrs.get('log_var_fal_final', 'N/D'):.4f}")
    # print(f"  log_var_fcl:  {study_arq.best_trial.user_attrs.get('log_var_fcl_final', 'N/D'):.4f}")

    # Traducimos a "precision" (peso relativo efectivo) para que sea más interpretable
    # precision_wmse = np.exp(-study_arq.best_trial.user_attrs.get('log_var_wmse_final', 0))
    # precision_fal  = np.exp(-study_arq.best_trial.user_attrs.get('log_var_fal_final', 0))
    # precision_fcl  = np.exp(-study_arq.best_trial.user_attrs.get('log_var_fcl_final', 0))
    # print(f"\n  Precisión efectiva (peso relativo, mayor = más influyente):")
    # print(f"    WMSE: {precision_wmse:.6f}")
    # print(f"    FAL:  {precision_fal:.6f}")
    # print(f"    FCL:  {precision_fcl:.6f}")

# --- Distribución sobre TODOS los trials completados, para ver el patrón general ---
# print("\nDistribución sobre todos los trials completados:")
# cols_logvar = ['number', 'value',
#                'user_attrs_log_var_wmse_final',
#                'user_attrs_log_var_fal_final',
#                'user_attrs_log_var_fcl_final']
# cols_logvar_disponibles = [c for c in cols_logvar if c in trials_df.columns]
# display(trials_df[trials_df['state'] == 'COMPLETE'][cols_logvar_disponibles].describe())

In [ ]:
d = trials_df[(trials_df['state'] == 'COMPLETE') & (trials_df['value'] < 0.1)].copy()

plt.figure(figsize=(6, 5))
sc = plt.scatter(d['value'], d['user_attrs_val_final'], c=np.log10(d['params_lr']), cmap='viridis', s=30)
plt.colorbar(sc, label='log10(lr)')
lim = [d['value'].min(), d['user_attrs_val_final'].max()]
plt.plot(lim, lim, 'k--', alpha=0.4)
for n in (8, 18, 37, 39, 60):                      # los candidatos a finalistas
    r = d[d['number'] == n].iloc[0]
    plt.annotate(str(n), (r['value'], r['user_attrs_val_final']), textcoords='offset points', xytext=(4, 4), fontsize=8)
plt.xlabel('mínimo del val_loss'); plt.ylabel('val_loss en la última época')
plt.grid(alpha=0.3); plt.tight_layout(); plt.show()

# Resultados

In [ ]:
def crear_cmap_lightdens(n_colores=256, ancho_transicion=30):
    """
    Colormap basado en 'hot_r', pero con el valor mínimo forzado a blanco
    puro. Los primeros `ancho_transicion` colores interpolan linealmente
    desde blanco hasta el color que 'hot_r' ya tiene en ese punto de la
    rampa, así la transición es continua (sin salto ni tramo plano).
    A partir de ahí, el colormap sigue siendo 'hot_r' sin modificar.
    """
    base = plt.get_cmap('hot_r', n_colores)
    colores = base(np.linspace(0, 1, n_colores))

    for i in range(ancho_transicion):
        t = i / (ancho_transicion - 1)
        colores[i] = (1 - t) * np.array([1, 1, 1, 1]) + t * colores[i]

    return LinearSegmentedColormap.from_list('lightdens_white', colores, N=n_colores)

CMAP_LIGHTDENS = crear_cmap_lightdens()

In [ ]:
def resumen_entrenamiento(checkpoint, nombre_modelo, train_loader, val_loader, batch_size):
    """
    Imprime un resumen del entrenamiento combinando datos del checkpoint
    (guardados durante el entrenamiento) con datos del DataLoader actual
    (que no se guardan en el .pth).
    """
    tiempo_total_seg = sum(checkpoint['historial_tiempo_epoch'])
    horas = int(tiempo_total_seg // 3600)
    minutos = int((tiempo_total_seg % 3600) // 60)
    tiempo_str = f"{horas}h {minutos}m" if horas > 0 else f"{minutos}m"

    print(f"-> Modelo Activo:  {nombre_modelo}")
    print(f"-> Épocas totales: {checkpoint['epochs_entrenadas']}")
    print(f"-> Learning Rate:  {checkpoint['lr']}")
    print(f"-> Batch Size:     {batch_size}")
    print(f"-> Archivos para entrenar (Train): {len(train_loader.dataset)}")
    print(f"-> Archivos para validar (Val): {len(val_loader.dataset)}")
    print(f"-> Batches por época (Train): {len(train_loader)}")
    print(f"-> Tiempo: {tiempo_str}")
    print(f"-> Mejor val_loss: {checkpoint['mejor_val_loss']:.6f} (época {checkpoint['mejor_epoca']})")
    print(f"-> Patience: {checkpoint['patience']}")

def generar_ruta_imagen(carpeta, prefijo, idx=None):
    """
    Genera la ruta de guardado. Si se provee un 'idx', lo incluye en el nombre.
    Si el archivo ya existe, le anexa un contador (_01, _02, etc.) para no pisarlo.
    """
    os.makedirs(carpeta, exist_ok=True)
    
    # Armamos el nombre base. Ej: "reconstruccion_unet_idx_42" o "curva_error_unet"
    if idx is not None:
        nombre_base = f"{prefijo}_idx_{idx}"
    else:
        nombre_base = prefijo
        
    ruta = os.path.join(carpeta, f"{nombre_base}.png")
    
    # Si no existe, usamos ese nombre directamente
    if not os.path.exists(ruta):
        return ruta
        
    # Si ya existe, empezamos a probar con un contador hasta encontrar uno libre
    contador = 1
    while True:
        ruta_con_contador = os.path.join(carpeta, f"{nombre_base}_{contador:02d}.png")
        if not os.path.exists(ruta_con_contador):
            return ruta_con_contador
        contador += 1

def graficar_curva_error(checkpoint, nombre_modelo='Modelo'):
    """
    Grafica la curva de MSE para train y validación a partir de un checkpoint cargado.
    """
    historial_train = checkpoint['historial_train_loss']
    historial_val   = checkpoint['historial_val_loss']
    epocas = range(1, len(historial_train) + 1)
    mejor_epoca    = checkpoint['mejor_epoca']
    mejor_val_loss = checkpoint['mejor_val_loss']

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.plot(epocas, historial_train, 'o-', label='Train Loss (HFACLV2.1)', color='steelblue')
    ax.plot(epocas, historial_val,   'o-', label='Val Loss (HFACLV2.1)',   color='#722F37')
    
    ax.scatter([mejor_epoca], [mejor_val_loss], s=180, marker='*',
               color='gold', edgecolor='black', zorder=5,
               label=f'Mejor época ({mejor_epoca})')
    ax.axvline(mejor_epoca, color='gray', linestyle='--', alpha=0.5)
    ax.annotate(f'{mejor_val_loss:.4f}',
                xy=(mejor_epoca, mejor_val_loss),
                xytext=(8, 10), textcoords='offset points',
                fontsize=9, fontweight='bold')
    
    ax.set_xlabel('Época')
    ax.set_ylabel('HFACLV2.1 Loss')
    ax.set_title(f'Curva de aprendizaje — {nombre_modelo}')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_guardado = generar_ruta_imagen('../images', f"curva_error_{nombre_limpio}")
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
    print(f"Gráfica de error guardada en: {ruta_guardado}")
    
    plt.show()
    
    mejor_epoca = historial_val.index(min(historial_val)) + 1
    print(f"Mejor val_loss: {min(historial_val):.6f} (época {mejor_epoca})")

def graficar_reconstruccion(modelo, dataset, nombre_modelo='Modelo', canal_entrada=0, idx=None):
    VAR_NOMBRES = ['MDBZ', 'HIELO', 'AGUA', 'CTT', 'LPI', 'WMAX']

    if idx is None:
        idx = random.randint(0, len(dataset) - 1)
    print(f"Timestep seleccionado: {idx} ({dataset.files[idx].name})")

    modelo.eval()
    X, y = dataset[idx]
    with torch.no_grad():
        pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

    y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
    y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())
    entrada = X[canal_entrada].numpy()

    # El real nunca puede superar 150 (clippeado en el dataset) — cualquier
    # exceso es ruido de float. La predicción sí puede superar 150 legítimamente.
    real_max = min(y_real.max(), 150.0)
    pred_max = round(float(y_pred.max()), 1)

    vmax_crudo = max(real_max, pred_max)
    if vmax_crudo <= 0:
        vmax_crudo = 1.0

    # Step "lindo" para los ticks de la colorbar
    paso_base = max(1, int(np.ceil(vmax_crudo / 7.0)))
    if paso_base >= 10:
        x = int(np.ceil(paso_base / 10.0)) * 10
    elif paso_base > 5:
        x = int(np.ceil(paso_base / 5.0)) * 5
    else:
        x = paso_base
    vmax_real = int(np.ceil(vmax_crudo / x)) * x
    ticks = np.arange(0, vmax_real + x, x)

    # Extensión del dominio y fondo (mismo criterio que la animación)
    ext_lon_min, ext_lon_max = float(np.min(LONS)), float(np.max(LONS))
    ext_lat_min, ext_lat_max = float(np.min(LATS)), float(np.max(LATS))

    def dibujar_fondo(ax):
        ax.set_extent([ext_lon_min, ext_lon_max, ext_lat_min, ext_lat_max], crs=crs.PlateCarree())
        ax.add_feature(cfeature.BORDERS, linewidth=0.8)
        ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
        ax.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')
        gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
        gl.top_labels = False
        gl.right_labels = False
        gl.xlabel_style = {'size': 8, 'color': 'gray'}
        gl.ylabel_style = {'size': 8, 'color': 'gray'}

    fig = plt.figure(figsize=(16, 5), facecolor='white')
    fig.suptitle(f'{nombre_modelo} — Timestep {idx}', fontsize=13)

    ax0 = fig.add_subplot(1, 3, 1, projection=CART_PROJ)
    dibujar_fondo(ax0)
    cf0 = ax0.contourf(LONS, LATS, entrada, transform=crs.PlateCarree(), cmap='Blues')
    ax0.set_title(f'Entrada: {VAR_NOMBRES[canal_entrada]}')
    fig.colorbar(cf0, ax=ax0, fraction=0.046, pad=0.04)

    ax1 = fig.add_subplot(1, 3, 2, projection=CART_PROJ)
    dibujar_fondo(ax1)
    cf1 = ax1.pcolormesh(LONS, LATS, y_real, transform=crs.PlateCarree(),
                          cmap=CMAP_LIGHTDENS, vmin=0, vmax=vmax_real, shading='auto')
    ax1.set_title('Real: LIGHTDENS')
    fig.colorbar(cf1, ax=ax1, fraction=0.046, pad=0.04, ticks=ticks)

    ax2 = fig.add_subplot(1, 3, 3, projection=CART_PROJ)
    dibujar_fondo(ax2)
    cf2 = ax2.pcolormesh(LONS, LATS, y_pred, transform=crs.PlateCarree(),
                          cmap=CMAP_LIGHTDENS, vmin=0, vmax=vmax_real, shading='auto')
    ax2.set_title(f'Predicción: {nombre_modelo}')
    fig.colorbar(cf2, ax=ax2, fraction=0.046, pad=0.04, ticks=ticks)

    plt.tight_layout()

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    ruta_guardado = generar_ruta_imagen('../images', f"reconstruccion_{nombre_limpio}", idx=idx)
    plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight', facecolor='white')
    print(f"Reconstrucción guardada en: {ruta_guardado}")

    plt.show()

    print(f"Real  — min: {y_real.min():.2f}, max: {y_real.max():.2f}, mean: {y_real.mean():.4f}")
    print(f"Pred  — min: {y_pred.min():.2f}, max: {y_pred.max():.2f}, mean: {y_pred.mean():.4f}")

def agrupar_indices_por_tormenta(dataset):
    """
    Agrupa los índices de `dataset` (TormentaDataset) en tormentas: tramos de
    archivos consecutivos en el tiempo (mismo paso que el resto del split).
    Como el dataset de val solo tiene los samples de las tormentas de
    validación (sin los timesteps de las tormentas de train intercaladas),
    un salto de tiempo entre dos archivos consecutivos marca el límite entre
    una tormenta y la siguiente. No depende de una lista de fechas mantenida
    a mano: si val_tormentas cambia, esto se ajusta solo.

    Devuelve dict id_tormenta -> lista de índices (ordenados), donde
    id_tormenta es la fecha/hora del primer archivo ('YYYY-MM-DD_HHMM').
    """
    n = len(dataset)
    if n == 0:
        return {}
    ts = []
    for idx in range(n):
        m = re.search(r'(\d{4}-\d{2}-\d{2})_(\d{2})-(\d{2})-(\d{2})', dataset.files[idx].name)
        fecha, h, mi, s = m.group(1), m.group(2), m.group(3), m.group(4)
        ts.append((idx, datetime.strptime(f"{fecha} {h}:{mi}:{s}", "%Y-%m-%d %H:%M:%S")))
    ts.sort(key=lambda x: x[1])
    if n == 1:
        return {f"{ts[0][1]:%Y-%m-%d_%H%M}": [ts[0][0]]}

    paso = min(b[1] - a[1] for a, b in zip(ts[:-1], ts[1:]))
    grupos = {}
    grupo_actual = [ts[0]]
    for a, b in zip(ts[:-1], ts[1:]):
        if b[1] - a[1] == paso:
            grupo_actual.append(b)
        else:
            clave = f"{grupo_actual[0][1]:%Y-%m-%d_%H%M}"
            grupos[clave] = [i for i, _ in grupo_actual]
            grupo_actual = [b]
    clave = f"{grupo_actual[0][1]:%Y-%m-%d_%H%M}"
    grupos[clave] = [i for i, _ in grupo_actual]
    return grupos

def animar_reconstruccion_por_tormenta(modelo, dataset, nombre_modelo='Model', fps=5, umbral_ruido=0.5):
    modelo.eval()

    grupos_por_tormenta = agrupar_indices_por_tormenta(dataset)
    rutas_generadas = []

    for tormenta_id, indices_tormenta in grupos_por_tormenta.items():
        print(f"\n=== Generating animation for {tormenta_id} ({len(indices_tormenta)} timesteps) ===")

        n_frames = len(indices_tormenta)
        predicciones_cache = []
        reales_cache = []

        for idx in tqdm(indices_tormenta, desc=f"Calculating range — {tormenta_id}"):
            X, y = dataset[idx]
            with torch.no_grad():
                pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

            y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
            y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())

            y_pred[y_pred < umbral_ruido] = np.nan
            y_real[y_real < umbral_ruido] = np.nan

            reales_cache.append(y_real)
            predicciones_cache.append(y_pred)

        reales_validos = np.concatenate([r[~np.isnan(r)] for r in reales_cache])
        preds_validos  = np.concatenate([p[~np.isnan(p)] for p in predicciones_cache])

        real_max = min(reales_validos.max(), 150.0) if reales_validos.size > 0 else 0.0
        pred_max = round(float(preds_validos.max()), 1) if preds_validos.size > 0 else 0.0

        vmax_crudo = max(real_max, pred_max)
        if vmax_crudo <= 0:
            vmax_crudo = 10.0

        paso_base = max(1, int(np.ceil(vmax_crudo / 7.0)))
        if paso_base >= 10:
            x = int(np.ceil(paso_base / 10.0)) * 10
        elif paso_base > 5:
            x = int(np.ceil(paso_base / 5.0)) * 5
        else:
            x = paso_base
        vmax = int(np.ceil(vmax_crudo / x)) * x
        ticks = np.arange(0, vmax + x, x)

        print(f"LIGHTDENS Range ({tormenta_id}): 0 - {vmax} (Step: {x})")

        fig, axes = plt.subplots(1, 2, figsize=(14, 6), subplot_kw={'projection': CART_PROJ})
        fig.suptitle(f'{nombre_modelo} — Validation storm {tormenta_id}', fontsize=15, y=0.98)

        ext_lon_min, ext_lon_max = float(np.min(LONS)), float(np.max(LONS))
        ext_lat_min, ext_lat_max = float(np.min(LATS)), float(np.max(LATS))

        def dibujar_fondo(ax):
            ax.set_facecolor('white')
            ax.set_extent([ext_lon_min, ext_lon_max, ext_lat_min, ext_lat_max], crs=crs.PlateCarree())
            ax.add_feature(cfeature.BORDERS, linewidth=0.8)
            ax.add_feature(cfeature.COASTLINE, linewidth=0.8)
            ax.add_feature(cfeature.STATES, linewidth=0.6, edgecolor='black')
            gl = ax.gridlines(draw_labels=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
            gl.top_labels = False
            gl.right_labels = False
            gl.xlabel_style = {'size': 8, 'color': 'gray'}
            gl.ylabel_style = {'size': 8, 'color': 'gray'}

        for ax in axes:
            dibujar_fondo(ax)

        pbar = tqdm(total=n_frames, desc="Rendering Animation")

        def update(frame_idx):
            for ax in axes:
                ax.cla()
                dibujar_fondo(ax)

            cf0 = axes[0].pcolormesh(LONS, LATS, reales_cache[frame_idx], transform=crs.PlateCarree(),
                                      cmap=CMAP_LIGHTDENS, vmin=0, vmax=vmax, shading='nearest')
            axes[0].set_title(f'Ground Truth: LIGHTDENS — Frame {frame_idx}')

            cf1 = axes[1].pcolormesh(LONS, LATS, predicciones_cache[frame_idx], transform=crs.PlateCarree(),
                                      cmap=CMAP_LIGHTDENS, vmin=0, vmax=vmax, shading='nearest')
            axes[1].set_title(f'Prediction: {nombre_modelo}')

            pbar.update(1)
            return cf0, cf1

        cf_inicial_real, cf_inicial_pred = update(0)

        fig.colorbar(cf_inicial_real, ax=axes[0], label='LIGHTDENS (flashes/column)', fraction=0.046, pad=0.04, ticks=ticks)
        fig.colorbar(cf_inicial_pred, ax=axes[1], label='LIGHTDENS (flashes/column)', fraction=0.046, pad=0.04, ticks=ticks)

        plt.tight_layout()

        ani = animation.FuncAnimation(fig, update, frames=range(n_frames), interval=200)

        carpeta_destino = "../videos&gifs"
        os.makedirs(carpeta_destino, exist_ok=True)
        nombre_limpio = nombre_modelo.replace(' ', '_').lower()
        nombre_archivo_salida = f"reconstruction_{nombre_limpio}_{tormenta_id}.gif"
        ruta_final = os.path.join(carpeta_destino, nombre_archivo_salida)

        ani.save(ruta_final, writer="pillow", fps=fps)
        pbar.close()
        plt.close(fig)

        print(f"Animation saved to {ruta_final}")
        rutas_generadas.append(ruta_final)

    return rutas_generadas
    
def mostrar_historial_tabular(checkpoint):
    """
    Toma el checkpoint de un modelo y muestra una tabla interactiva con 
    el historial de pérdidas (Train y Val) época por época.
    Resalta en verde la época con el menor error de validación.
    """
    # 1. Extraemos las listas del checkpoint
    historial_train = checkpoint['historial_train_loss']
    historial_val   = checkpoint['historial_val_loss']
    epocas = range(1, len(historial_train) + 1)
    
    # 2. Armamos un DataFrame de Pandas
    df = pd.DataFrame({
        'Época': epocas,
        'Train Loss (FACL)': historial_train,
        'Val Loss (FACL)': historial_val
    })
    
    # 3. Calculamos la diferencia con la época anterior para ver si mejoró
    mejor_hasta_ayer = df['Val Loss (FACL)'].shift(1).cummin().fillna(float('inf'))
    df['¿Mejoró?'] = (df['Val Loss (FACL)'] < mejor_hasta_ayer).apply(lambda x: '🟢 Sí' if x else '🔴 No')
    df.loc[0, '¿Mejoró?'] = '🟢 Sí'
    
    # 4. Buscamos cuál fue la mejor época absoluta
    mejor_val_loss = min(historial_val)
    
    # 5. Función para pintar la fila ganadora de verde
    def resaltar_mejor_epoca(fila):
        if fila['Val Loss (FACL)'] == mejor_val_loss:
            return ['background-color: #d4edda; color: #155724; font-weight: bold'] * len(fila)
        else:
            return [''] * len(fila)
            
    # 6. Aplicamos el formato (dejamos 6 decimales) y mostramos la tabla
    tabla_estilizada = df.style.apply(resaltar_mejor_epoca, axis=1) \
                               .format({'Train Loss (FACL)': "{:.6f}", 
                                        'Val Loss (FACL)': "{:.6f}"}) \
                               .hide(axis="index")
    
    print("HISTORIAL DE ENTRENAMIENTO TABULAR")
    print("-" * 40)
    display(tabla_estilizada)

    return df

def calcular_data_range_global(dataset):
    """
    Calcula el rango (max - min) de LIGHTDENS real sobre TODO el dataset.
    """
    max_global = float("-inf")
    min_global = float("inf")
    for idx in tqdm(range(len(dataset)), desc="Calculando data_range global"):
        _, y = dataset[idx]
        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
        max_global = max(max_global, y_real.max())
        min_global = min(min_global, y_real.min())
    return max_global - min_global

def calcular_metricas_timestep(y_real, y_pred, data_range=None):
    """
    Calcula un set de métricas de verificación comparando la predicción
    contra LIGHTDENS real, para un único timestep. Todas las métricas
    se evalúan solo sobre los píxeles con actividad eléctrica (observada
    o predicha).
    """
    mascara = (y_real > 0) | (y_pred > 0)
    actividad_total = y_real.sum()  # se mide sobre el dominio completo, no filtrado

    if data_range is None:
        data_range = max(y_real.max(), y_pred.max()) - min(y_real.min(), y_pred.min())

    # Mapa de SSIM local, calculado sobre el campo completo
    if data_range > 0:
        _, mapa_ssim = ssim(y_real, y_pred, data_range=data_range, full=True)
    else:
        mapa_ssim = np.ones_like(y_real)

    # Si no hay actividad observada NI predicha, devolvemos NaN para que pandas lo ignore al graficar
    if mascara.sum() == 0:
        return {'mse': np.nan, 'rmse': np.nan, 'mae': np.nan, 'ssim': np.nan,
                'pod': np.nan, 'far': np.nan, 'csi': np.nan,
                'actividad': actividad_total}

    # Filtramos por la máscara
    y_real_m = y_real[mascara]
    y_pred_m = y_pred[mascara]

    # Métricas de error continuo
    error = y_real_m - y_pred_m
    mse = np.mean(error ** 2)
    rmse = np.sqrt(mse)
    mae = np.mean(np.abs(error))
    valor_ssim = mapa_ssim[mascara].mean()

    # Cálculo de las 3 Titánides Categóricas (POD, FAR, CSI)
    obs_bin = (y_real_m > 0)
    pred_bin = (y_pred_m > 0)

    hits = (obs_bin & pred_bin).sum()
    misses = (obs_bin & ~pred_bin).sum()
    false_alarms = (~obs_bin & pred_bin).sum()

    pod = hits / (hits + misses) if (hits + misses) > 0 else np.nan
    far = false_alarms / (hits + false_alarms) if (hits + false_alarms) > 0 else np.nan
    csi = hits / (hits + misses + false_alarms) if (hits + misses + false_alarms) > 0 else np.nan

    return {
        'mse': mse, 'rmse': rmse, 'mae': mae, 'ssim': valor_ssim,
        'pod': pod, 'far': far, 'csi': csi,
        'actividad': actividad_total
    }

def evaluar_dataset(modelo, dataset, data_range=None):
    modelo.eval()
    registros = []

    if data_range is None:
        data_range = calcular_data_range_global(dataset)
        print(f"Data range global (LIGHTDENS): {data_range:.2f}")

    grupos_por_tormenta = agrupar_indices_por_tormenta(dataset)
    tormenta_de_idx = {idx: tid for tid, idxs in grupos_por_tormenta.items() for idx in idxs}

    for idx in tqdm(range(len(dataset)), desc="Calculando métricas"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
        y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())

        metricas = calcular_metricas_timestep(y_real, y_pred, data_range=data_range)
        metricas['idx'] = idx
        metricas['tormenta'] = tormenta_de_idx[idx]

        registros.append(metricas)

    return pd.DataFrame(registros)

def graficar_metrica(df_metricas, columna, nombre_modelo='Model', ylabel=None, mostrar_actividad=False):
    """
    Plots the evolution of a verification metric across the validation
    timesteps, coloring each validation storm differently and mapping
    storm ids in the legend (English labels).
    """
    # Determinamos qué columnas necesitamos extraer
    columnas_req = ['idx', columna, 'tormenta']
    if mostrar_actividad and 'actividad' in df_metricas.columns:
        columnas_req.append('actividad')
        
    serie = df_metricas[columnas_req].dropna(subset=[columna])

    with plt.style.context('seaborn-v0_8-darkgrid'):
        fig, ax = plt.subplots(figsize=(11, 5), facecolor='white')

        tormentas_unicas = serie['tormenta'].unique()
        colores = plt.cm.tab10(np.linspace(0, 1, len(tormentas_unicas)))

        # NUEVO: Crear eje secundario para la actividad eléctrica (fondo)
        ax2 = None
        if mostrar_actividad and 'actividad' in df_metricas.columns:
            ax2 = ax.twinx()
            ax2.bar(serie['idx'], serie['actividad'], width=1.0, color='gray', alpha=0.25, 
                    label='Electrical Activity (Total Flashes)')
            ax2.set_ylabel('Total Flashes (Ground Truth)', color='gray', fontweight='bold')
            ax2.tick_params(axis='y', labelcolor='gray')
            ax2.grid(False) # Apagamos la grilla secundaria para no ensuciar la principal

        # Graficamos la métrica principal
        for i, tormenta in enumerate(tormentas_unicas):
            grupo = serie[serie['tormenta'] == tormenta]
            ax.plot(
                grupo['idx'],
                grupo[columna],
                'o-',
                color=colores[i],
                markersize=4,
                linewidth=1.2,
                label=f"Storm {tormenta}",
                zorder=3 # Asegura que la línea quede por encima de las barras
            )

        media = serie[columna].mean()
        ax.axhline(
            media,
            color='black',
            linestyle='--',
            alpha=0.8,
            linewidth=1.5,
            label=f'Overall Mean: {media:.3f}',
            zorder=3
        )

        ax.set_xlabel('Validation Timestep')
        ax.set_ylabel(ylabel or columna.upper(), fontweight='bold')
        ax.set_title(f'{ylabel or columna.upper()} — {nombre_modelo}')

        # Combinar leyendas de ambos ejes si ax2 existe
        # (El bbox_to_anchor se corre a 1.15 para no superponerse con los números del eje secundario)
        if ax2:
            lines_1, labels_1 = ax.get_legend_handles_labels()
            lines_2, labels_2 = ax2.get_legend_handles_labels()
            ax.legend(lines_1 + lines_2, labels_1 + labels_2, 
                      bbox_to_anchor=(1.15, 1), loc='upper left', frameon=True, facecolor='white')
        else:
            ax.legend(bbox_to_anchor=(1.02, 1), loc='upper left', frameon=True, facecolor='white')
            
        plt.tight_layout()

        nombre_limpio = nombre_modelo.replace(' ', '_').lower()
        ruta_guardado = generar_ruta_imagen('../images', f"{columna}_{nombre_limpio}")

        plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Plot for {columna.upper()} saved to: {ruta_guardado}")

        plt.show()

    return media

def graficar_mse(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'mse', nombre_modelo, ylabel='MSE')

def graficar_rmse(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'rmse', nombre_modelo, ylabel='RMSE')

def graficar_mae(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'mae', nombre_modelo, ylabel='MAE')

def graficar_ssim(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'ssim', nombre_modelo, ylabel='SSIM', mostrar_actividad=True)

def graficar_pod(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'pod', nombre_modelo, ylabel='Probability of Detection (POD)', mostrar_actividad=True)

def graficar_far(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'far', nombre_modelo, ylabel='False Alarm Ratio (FAR)', mostrar_actividad=True)

def graficar_csi(df, nombre_modelo='Model'):
    return graficar_metrica(df, 'csi', nombre_modelo, ylabel='Critical Success Index (CSI)', mostrar_actividad=True)

def calcular_r2_global(modelo, dataset, nombre_modelo='Model', ignorar_borde=0):
    """
    Calculates R² as a global metric over all active pixels
    in the dataset, for a single model.
    """
    modelo.eval()
    y_real_total = []
    y_pred_total = []

    for idx in tqdm(range(len(dataset)), desc=f"Calculating R² — {nombre_modelo}"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
        y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())

        if ignorar_borde > 0:
            y_real = y_real[ignorar_borde:-ignorar_borde, ignorar_borde:-ignorar_borde]
            y_pred = y_pred[ignorar_borde:-ignorar_borde, ignorar_borde:-ignorar_borde]

        mascara = (y_real > 0) | (y_pred > 0)
        if mascara.sum() > 0:
            y_real_total.append(y_real[mascara])
            y_pred_total.append(y_pred[mascara])

    y_real_all = np.concatenate(y_real_total)
    y_pred_all = np.concatenate(y_pred_total)

    ss_res = np.sum((y_real_all - y_pred_all) ** 2)
    ss_tot = np.sum((y_real_all - y_real_all.mean()) ** 2)
    r2 = 1 - ss_res / ss_tot

    edge_label = f" (ignoring {ignorar_borde}px edge)" if ignorar_borde > 0 else " (full image)"
    print(f"R² {nombre_modelo}{edge_label} — over {len(y_real_all):,} active pixels: {r2:.4f}")

    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    sufijo_borde = f"_noedge{ignorar_borde}" if ignorar_borde > 0 else "_full"
    ruta_log = f"../modelos/r2_{nombre_limpio}{sufijo_borde}.txt"
    
    with open(ruta_log, 'w', encoding='utf-8') as f:
        f.write(f"Model: {nombre_modelo}\n")
        f.write(f"Edge crop: {ignorar_borde}px\n")
        f.write(f"Active pixels evaluated: {len(y_real_all):,}\n")
        f.write(f"R²: {r2:.6f}\n")
    print(f"Result saved to: {ruta_log}")

    residuos = y_pred_all - y_real_all

    with plt.style.context('seaborn-v0_8-darkgrid'):
        # Añadido facecolor='white' a la figura también para asegurar consistencia
        fig, ax = plt.subplots(figsize=(9, 5), facecolor='white')
        ax.hist(residuos, bins=100, color='#722F37', edgecolor='none', alpha=0.85)
        ax.axvline(0, color='black', linestyle='--', linewidth=1, alpha=0.7, label='Error = 0')
        ax.axvline(residuos.mean(), color='gold', linestyle='-', linewidth=2,
                   label=f'Mean bias: {residuos.mean():+.2f}')
        
        ax.set_xlim(-100, 100)
        
        ax.set_xlabel('Residuals (Predicted − Actual)')
        ax.set_ylabel('Frequency')
        ax.set_title(f'Residuals Distribution — {nombre_modelo}{edge_label}')
        ax.legend()
        plt.tight_layout()

        ruta_hist = f"../images/residuals_{nombre_limpio}{sufijo_borde}.png"
        # Facecolor white al exportar
        plt.savefig(ruta_hist, dpi=300, bbox_inches='tight', facecolor='white')
        print(f"Histogram saved to: {ruta_hist}")
        plt.show()

    return r2

def graficar_picos_de_error(df_metricas, modelo, dataset, nombre_modelo='Modelo', top_n=3, metrica='mse', canal_entrada=0):
    """
    Encuentra los timesteps (índices) donde el modelo tuvo el mayor error 
    (los picos más altos) y grafica la reconstrucción para entender qué pasó.
    
    Parámetros
    ----------
    df_metricas   : DataFrame devuelto por evaluar_dataset().
    modelo        : modelo PyTorch ya cargado y en eval mode.
    dataset       : TormentaDataset (val_dataset).
    nombre_modelo : str, para el título.
    top_n         : int, cantidad de picos a mostrar (por defecto los 3 peores).
    metrica       : str, la columna del df a usar para buscar los picos ('mse', 'mae', etc).
    canal_entrada : int, índice del canal a mostrar en la gráfica de entrada.
        Percentil de los valores activos (LIGHTDENS > 0) utilizado como
        límite superior de la escala de color en las reconstrucciones.
        Valores menores achican más la escala y resaltan mejor la estructura
        espacial a costa de saturar los picos más intensos.
    """
    # Verificamos que la métrica exista en el DataFrame
    if metrica not in df_metricas.columns:
        raise ValueError(f"La métrica '{metrica}' no se encuentra en el DataFrame.")

    # Calculamos la media para dar un poco de contexto de "qué tan grave" es el pico
    media_metrica = df_metricas[metrica].mean()
    
    # Extraemos las 'top_n' filas con los valores más altos de la métrica
    df_picos = df_metricas.nlargest(top_n, metrica)
    
    print(f"{"="*60}")
    print(f"🔍 ANALIZANDO LOS {top_n} PEORES CASOS SEGÚN {metrica.upper()}")
    print(f"Media global de {metrica.upper()} en este dataset: {media_metrica:.4f}")
    print(f"{"="*60}\n")
    
    # Iteramos sobre esas peores filas
    for ranking, (_, fila) in enumerate(df_picos.iterrows(), start=1):
        idx_pico = int(fila['idx'])
        valor_pico = fila[metrica]
        
        # Calculamos cuántas veces peor es este pico en comparación a la media
        multiplo = valor_pico / media_metrica if media_metrica > 0 else 0
        
        print(f"⚠️ PICO #{ranking} | Timestep: {idx_pico}")
        print(f"-> {metrica.upper()}: {valor_pico:.4f} (El error es {multiplo:.1f} veces la media global)")
        
        # Graficamos el índice extraído
        graficar_reconstruccion(
            modelo=modelo,
            dataset=dataset,
            nombre_modelo=nombre_modelo,
            canal_entrada=canal_entrada,
            idx=idx_pico,
        )
        print("-" * 60)
        print("\n")

def resumen_pesos_loss(criterion):
    """
    Muestra los pesos aprendidos de Uncertainty Weighting al final
    del entrenamiento, convertidos a sigma^2 para interpretación directa.

    Valores más chicos de sigma^2 indican que ese término tuvo mayor
    peso relativo (más "confianza") durante el entrenamiento.
    """
    log_vars = {
        'WMSE': criterion.log_var_wmse.item(),
        'FAL':  criterion.log_var_fal.item(),
        'FCL':  criterion.log_var_fcl.item(),
    }

    print("PESOS APRENDIDOS (Uncertainty Weighting)")
    print("-" * 45)
    for nombre, log_var in log_vars.items():
        sigma2 = np.exp(log_var)
        precision = np.exp(-log_var)
        print(f"  {nombre:5s} — log_var: {log_var:+.4f} | σ²: {sigma2:.4f} | peso (precisión): {precision:.4f}")

    return log_vars

def graficar_distribucion_curvas(modelo, dataset, nombre_modelo='Modelo', bin_w=1.0, sigma=2.0):
    """Curvas de densidad (eje Y log) de Observed vs Predicted, sobre TODOS los píxeles."""
    modelo.eval()
    xmax = float(dataset.desnormalizar_y(torch.tensor(1.0)))   # máximo posible de la escala
    bins = np.arange(0, xmax + bin_w, bin_w)
    centros = 0.5 * (bins[:-1] + bins[1:])
    cnt_real = np.zeros(len(centros), dtype=np.float64)
    cnt_pred = np.zeros(len(centros), dtype=np.float64)
    pn_min, pn_max, pn_sum, n = np.inf, -np.inf, 0.0, 0   # stats de la salida normalizada

    for idx in tqdm(range(len(dataset)), desc=f"Recolectando — {nombre_modelo}"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_norm = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy().ravel())
        y_pred = np.round(dataset.desnormalizar_y(pred_norm).numpy().ravel())
        cnt_real += np.histogram(y_real, bins=bins)[0]
        cnt_pred += np.histogram(y_pred, bins=bins)[0]

        pn = pred_norm.numpy().ravel()
        pn_min, pn_max = min(pn_min, pn.min()), max(pn_max, pn.max())
        pn_sum += pn.sum(); n += pn.size

    fig, ax = plt.subplots(figsize=(7, 5))
    for cnt, lbl, c in [(cnt_real, "Observed", "tab:blue"),
                        (cnt_pred, "Predicted", "tab:orange")]:
        if cnt.sum() == 0:
            continue
            
        dens = cnt / (cnt.sum() * bin_w)
        
        # --- APLICAR SUAVIZADO AQUÍ ---
        if sigma > 0:
            dens = gaussian_filter1d(dens, sigma=sigma)
            
        m = dens > 0
        ax.plot(centros[m], dens[m], label=lbl, color=c, lw=1.2)

    ax.set_yscale("log")
    ax.set_xlabel("flashes / columna")
    ax.set_ylabel("Probability Density")
    ax.set_title(f"Lightning distribution — {nombre_modelo}")
    ax.legend(); ax.grid(alpha=0.3)
    plt.tight_layout()
    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    plt.savefig(f"../images/dist_curvas_{nombre_limpio}.png", dpi=300, bbox_inches='tight')
    plt.show()

    print(f"Salida normalizada del modelo — min: {pn_min:.4f}, max: {pn_max:.4f}, media: {pn_sum/n:.4f}")
    return centros, cnt_real, cnt_pred


def graficar_distribucion_curvas_por_tormenta(modelo, dataset, nombre_modelo='Modelo', bin_w=1.0, sigma=2.0):
    """
    Curvas de densidad (eje Y log) de Observed vs Predicted, una por cada
    tormenta de `dataset` (detectadas automáticamente por salto temporal).
    """
    modelo.eval()
    xmax = float(dataset.desnormalizar_y(torch.tensor(1.0)))
    bins = np.arange(0, xmax + bin_w, bin_w)
    centros = 0.5 * (bins[:-1] + bins[1:])

    grupos = agrupar_indices_por_tormenta(dataset)
    resultados = {}

    for tormenta_id, indices in grupos.items():
        cnt_real = np.zeros(len(centros), dtype=np.float64)
        cnt_pred = np.zeros(len(centros), dtype=np.float64)
        pn_min, pn_max, pn_sum, n = np.inf, -np.inf, 0.0, 0

        for idx in tqdm(indices, desc=f"Recolectando — {tormenta_id}"):
            X, y = dataset[idx]
            with torch.no_grad():
                pred_norm = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

            y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy().ravel())
            y_pred = np.round(dataset.desnormalizar_y(pred_norm).numpy().ravel())
            cnt_real += np.histogram(y_real, bins=bins)[0]
            cnt_pred += np.histogram(y_pred, bins=bins)[0]

            pn = pred_norm.numpy().ravel()
            pn_min, pn_max = min(pn_min, pn.min()), max(pn_max, pn.max())
            pn_sum += pn.sum(); n += pn.size

        fig, ax = plt.subplots(figsize=(7, 5))
        for cnt, lbl, c in [(cnt_real, "Observed", "tab:blue"),
                            (cnt_pred, "Predicted", "tab:orange")]:
            if cnt.sum() == 0:
                continue
                
            dens = cnt / (cnt.sum() * bin_w)
            
            # --- APLICAR SUAVIZADO AQUÍ ---
            if sigma > 0:
                dens = gaussian_filter1d(dens, sigma=sigma)
                
            m = dens > 0
            ax.plot(centros[m], dens[m], label=lbl, color=c, lw=1.2)

        ax.set_yscale("log")
        ax.set_xlabel("flashes / columna")
        ax.set_ylabel("Probability Density")
        ax.set_title(f"Lightning distribution — {nombre_modelo} — Storm {tormenta_id}")
        ax.legend(); ax.grid(alpha=0.3)
        plt.tight_layout()

        nombre_limpio = nombre_modelo.replace(' ', '_').lower()
        ruta = f"../images/dist_curvas_{nombre_limpio}_{tormenta_id}.png"
        plt.savefig(ruta, dpi=300, bbox_inches='tight')
        plt.show()

        print(f"Storm {tormenta_id} — salida normalizada: min={pn_min:.4f}, max={pn_max:.4f}, media={pn_sum/n:.4f}")

        resultados[tormenta_id] = {
            "centros": centros, "cnt_real": cnt_real, "cnt_pred": cnt_pred,
            "pn_min": pn_min, "pn_max": pn_max, "pn_mean": pn_sum / n,
        }

    return resultados

def calcular_matriz_fss_dataset(modelo, dataloader, dataset,
                                umbrales=(1, 5, 15, 30),
                                ventanas=(1, 3, 5, 15)):
    """
    FSS global en GPU, evaluado solo en los puntos con actividad eléctrica
    observada (LIGHTDENS > 0) ya sea por el modelo o por WRF-Elec. Acumula 
    sum((obs_frac - pred_frac)^2) y sum(obs_frac^2 + pred_frac^2) sobre 
    esos puntos y divide recién al final. Devuelve un np.ndarray de forma
    (len(ventanas), len(umbrales)).
    """
    assert all(v % 2 == 1 for v in ventanas), "Las ventanas deben ser impares (padding simétrico)."

    modelo.eval()
    U, V = len(umbrales), len(ventanas)
    umb = torch.tensor(umbrales, device=device, dtype=torch.float32).view(1, U, 1, 1)

    num = torch.zeros(V, U, device=device, dtype=torch.float64)
    den = torch.zeros(V, U, device=device, dtype=torch.float64)

    with torch.no_grad():
        for X, y in tqdm(dataloader, desc="FSS"):
            X, y = X.to(device, non_blocking=True), y.to(device, non_blocking=True)
            p = modelo(X)

            y_real = torch.round(dataset.desnormalizar_y(y))
            y_pred = torch.round(dataset.desnormalizar_y(p))

            # Máscara de puntos con actividad eléctrica observada: (B, 1, H, W)
            mask = ((y_real > 0) | (y_pred > 0)).float()

            obs_bin  = (y_real >= umb).float()
            pred_bin = (y_pred >= umb).float()

            for i, v in enumerate(ventanas):
                # Las fracciones se calculan sobre el campo completo (con los vecinos)
                obs_frac  = F.avg_pool2d(obs_bin,  kernel_size=v, stride=1, padding=v // 2)
                pred_frac = F.avg_pool2d(pred_bin, kernel_size=v, stride=1, padding=v // 2)

                # ...pero el error se suma solo donde hay actividad (mask se broadcastea sobre U)
                num[i] += (((obs_frac - pred_frac) ** 2) * mask).sum(dim=(0, 2, 3)).double()
                den[i] += ((obs_frac ** 2 + pred_frac ** 2) * mask).sum(dim=(0, 2, 3)).double()

    fss = torch.where(den == 0, torch.ones_like(den), 1.0 - num / den)
    return fss.cpu().numpy()


def graficar_heatmap_fss(matriz, umbrales=(1, 5, 15, 30), ventanas=(1, 3, 5, 15),
                         nombre_modelo='Modelo'):
    plt.figure(figsize=(7, 5), facecolor='white')
    cmap = sns.diverging_palette(10, 130, as_cmap=True)
    ax = sns.heatmap(
        matriz, annot=True, fmt=".2f", cmap=cmap,
        vmin=0.0, vmax=1.0, center=0.5,
        xticklabels=umbrales, yticklabels=ventanas,
        linewidths=1, linecolor='white',
        cbar_kws={'label': 'Fractional Skill Score (FSS)'}
    )
    ax.invert_yaxis()
    plt.xlabel('Umbral de Intensidad (flashes/columna)', fontweight='bold')
    plt.ylabel('Escala Espacial / Ventana (px)', fontweight='bold')
    plt.title(f'Matriz FSS: Tolerancia Espacial vs Severidad — {nombre_modelo}', pad=15)
    plt.tight_layout()
    nombre_limpio = nombre_modelo.replace(' ', '_').lower()
    plt.savefig(f"../images/fss_heatmap_{nombre_limpio}.png", dpi=300, bbox_inches='tight')
    plt.show()

def evaluar_ssim_ventanas(modelo, dataset, win_sizes=[3, 5, 7, 11, 15], data_range=None):
    """
    Evalúa el dataset con el modelo actual calculando SSIM para distintas
    ventanas espaciales (win_size). El SSIM se calcula sobre el campo
    completo (para no romper la ventana local) y se promedia solo sobre
    los píxeles con actividad eléctrica (observada o predicha).
    """
    modelo.eval()
    registros = []

    if data_range is None:
        data_range = calcular_data_range_global(dataset)
        print(f"Data range global (LIGHTDENS): {data_range:.2f}")

    grupos_por_tormenta = agrupar_indices_por_tormenta(dataset)
    tormenta_de_idx = {idx: tid for tid, idxs in grupos_por_tormenta.items() for idx in idxs}

    for idx in tqdm(range(len(dataset)), desc="Calculando SSIM multi-ventana"):
        X, y = dataset[idx]
        with torch.no_grad():
            pred_log = modelo(X.unsqueeze(0).to(device)).cpu().squeeze()

        y_real = np.round(dataset.desnormalizar_y(y.squeeze()).numpy())
        y_pred = np.round(dataset.desnormalizar_y(pred_log).numpy())

        mascara = (y_real > 0) | (y_pred > 0)

        registro = {
            'idx': idx,
            'tormenta': tormenta_de_idx[idx],
            'actividad': y_real.sum()
        }

        for w in win_sizes:
            if data_range > 0 and mascara.sum() > 0:
                _, mapa_ssim = ssim(y_real, y_pred, data_range=data_range, win_size=w, full=True)
                registro[f'ssim_{w}x{w}'] = mapa_ssim[mascara].mean()
            elif data_range <= 0:
                registro[f'ssim_{w}x{w}'] = 1.0
            else:
                # Sin actividad ni observada ni predicha: nada que promediar
                registro[f'ssim_{w}x{w}'] = np.nan

        registros.append(registro)

    return pd.DataFrame(registros)

def graficar_sensibilidad_ssim_ventanas(df_resultados, win_sizes=[3, 5, 7, 11, 15], nombre_modelo='Modelo'):
    # Transformamos el DataFrame a formato largo (melt) para seaborn
    columnas_ssim = [f'ssim_{w}x{w}' for w in win_sizes]
    df_long = df_resultados.melt(
        id_vars=['idx', 'tormenta'], 
        value_vars=columnas_ssim,
        var_name='Ventana', 
        value_name='SSIM'
    )

    with plt.style.context('seaborn-v0_8-darkgrid'):
        fig, ax = plt.subplots(figsize=(10, 5), facecolor='white')
        
        sns.boxplot(
            data=df_long, 
            x='Ventana', 
            y='SSIM', 
            ax=ax, 
            palette='Blues',
            showmeans=True,
            meanprops={"marker":"o", "markerfacecolor":"red", "markeredgecolor":"red"}
        )

        # Imprimir las medias exactas en pantalla
        print("--- Medias de SSIM por ventana ---")
        for col in columnas_ssim:
            print(f"{col}: {df_resultados[col].mean():.4f}")

        ax.set_title(f"Análisis de Sensibilidad de SSIM según el Tamaño de Ventana — {nombre_modelo}", fontweight='bold', fontsize=13)
        ax.set_ylabel("SSIM", fontweight='bold')
        ax.set_xlabel("Tamaño de Ventana (win_size)", fontweight='bold')
        
        plt.tight_layout()
        
        # Limpiamos el nombre del modelo para usarlo en el nombre del archivo
        nombre_limpio = nombre_modelo.replace(' ', '_').lower()
        ruta_guardado = f'../images/sensibilidad_ssim_ventanas_{nombre_limpio}.png'
        
        plt.savefig(ruta_guardado, dpi=300, bbox_inches='tight')
        print(f"Plot saved to: {ruta_guardado}")
        
        plt.show()

In [ ]:
ruta = '../modelos/etapa_B_Ext_Clip_U-Net_FACL/B_trial0039_Ext Clip U-Net FACL.pth'   # cambiá el número de trial
checkpoint = torch.load(ruta, map_location=device, weights_only=False)   # checkpoint propio, de confianza
modelo.load_state_dict(checkpoint['model_state_dict'])   # 'model_state_dict' para la última época
modelo.eval()
resumen_entrenamiento(checkpoint, NOMBRE_MODELO, train_loader, val_loader, checkpoint['batch_size'])

In [ ]:
graficar_curva_error(checkpoint, nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_reconstruccion(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO, idx=None)

In [ ]:
rutas = animar_reconstruccion_por_tormenta(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO)

In [ ]:
df_historial = mostrar_historial_tabular(checkpoint)
ruta_txt = f"../modelos/historial_{NOMBRE_MODELO}.txt"

with open(ruta_txt, 'w', encoding='utf-8') as f:
    f.write("=== HISTORIAL DE ENTRENAMIENTO ===\n")
    f.write(df_historial.to_string(index=False))
    
print(f"¡Guardado en {ruta_txt}!")

In [ ]:
data_range_val = calcular_data_range_global(val_dataset)
df_metricas = evaluar_dataset(modelo, val_dataset, data_range=data_range_val)

In [ ]:
graficar_mse(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_rmse(df_metricas, nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_mae(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_pod(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_far(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
graficar_csi(df_metricas,  nombre_modelo=NOMBRE_MODELO)

In [ ]:
# Ventana por default de 7x7
graficar_ssim(df_metricas, nombre_modelo=NOMBRE_MODELO)

In [ ]:
# Ventanas a probar (siempre impares)
ventanas_a_probar = [3, 5, 7, 11, 15]

df_ssim_ventanas = evaluar_ssim_ventanas(
    modelo=modelo, 
    dataset=val_dataset, 
    win_sizes=ventanas_a_probar
)

graficar_sensibilidad_ssim_ventanas(df_ssim_ventanas, ventanas_a_probar, NOMBRE_MODELO)

In [ ]:
r2_completo   = calcular_r2_global(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO, ignorar_borde=0)
r2_sin_bordes = calcular_r2_global(modelo, val_dataset, nombre_modelo=NOMBRE_MODELO, ignorar_borde=10)

print(f"\nDiferencia: {r2_sin_bordes - r2_completo:+.4f}")

In [ ]:
matriz_fss = calcular_matriz_fss_dataset(modelo, val_loader, val_dataset)
graficar_heatmap_fss(matriz_fss, nombre_modelo=NOMBRE_MODELO)

In [ ]:
centros, cnt_real, cnt_pred = graficar_distribucion_curvas(modelo, val_dataset, NOMBRE_MODELO)

In [ ]:
resultados = graficar_distribucion_curvas_por_tormenta(modelo, val_dataset, NOMBRE_MODELO)

In [ ]:
graficar_picos_de_error(df_metricas, modelo, val_dataset, nombre_modelo=NOMBRE_MODELO)

In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================
# CELDA PARA Entrenamiento_Ext_Clip_U-Net_FACL.ipynb  (se puede correr
# en el JupyterHub de Serafín o en Clementina; hace falta una GPU)
#
# Qué hace: obtiene los pesos de la Ext Clip U-Net FACL en su "mejor
# época según L_val" (la 8), que la reproducción de las curvas no
# guardó, para mostrar en Resultados cómo cambia la serie temporal de
# validación según qué pérdida de validación elige la época.
#   PARTE 1: reentrena el ensayo 39 de la Fase 2 con exactamente lo
#            mismo que celda_curvas_fal_fcl_trial39.py (semillas,
#            hiperparámetros leídos de Optuna, arquitectura del
#            ensayo 106), pero SOLO LAS PRIMERAS 8 ÉPOCAS, con el
#            cronograma de las 140 (T_max, FACL y probabilidad de
#            usar la FCL), y guarda los pesos de la época 8.
#            En Clementina dura ~1 h 10 min (8 épocas de ~8,3 min);
#            en el JupyterHub tardará más (la Ext U-Net Comb
#            V2.1, de 8 millones de parámetros, tardó ~25 min por
#            época ahí), así que contá con unas 3 o 4 horas.
#   PARTE 2: estima con ese modelo las 2.271 muestras de validación y
#            guarda resultados_validacion_epoca8.npz (el archivo
#            anterior más la clave `pred_ext_epoca8`).
#
# Antes de correrla, ejecutá las celdas de: Dependencias, semillas
# (SEED = 42), UNet_WRF, FACL, DATASETS (incluye TormentaDataset) e
# `import optuna`. Copiá resultados_validacion.npz a la carpeta
# desde la que corrés (o cambiá RUTA_NPZ). Si se corta, volvé a
# correrla: sigue desde la última época completa.
#
# Salidas: ../modelos/etapa_B_Ext_Clip_U-Net_FACL/
#              B_trial0039_repro_epoca8.pth
#          resultados_validacion_epoca8.npz   <- mandame este
# =====================================================================
import glob
import json
import os
import time
import optuna

TRIAL_ARQ, TRIAL_ENT = 106, 39
BATCH_VAL = 16
EPOCA_FINAL = 8      # se entrena con el cronograma de 140 épocas
SALIDA = f'../estudios/trial{TRIAL_ENT:04d}_hasta_epoca8.json'
CKPT = (f'../modelos/etapa_B_Ext_Clip_U-Net_FACL/'
        f'B_trial{TRIAL_ENT:04d}_repro_epoca8.pth')


def buscar_estudio(patron):
    """Primer estudio de Optuna cuyo .db coincide con el patrón."""
    for db in sorted(glob.glob(f'../estudios/{patron}')):
        storage = f'sqlite:///{db}'
        for resumen in optuna.get_all_study_summaries(storage):
            print(f'  {os.path.basename(db)} -> {resumen.study_name}')
            return optuna.load_study(study_name=resumen.study_name,
                                     storage=storage)
    raise FileNotFoundError(f'No encontré ../estudios/{patron}')


print('Estudios encontrados:')
est_arq = buscar_estudio('optuna_arquitectura_*Ext Clip U-Net FACL*.db')
est_ent = buscar_estudio('optuna_entrenamiento_*Ext Clip U-Net FACL*.db')
pa = est_arq.trials[TRIAL_ARQ].params
pe = est_ent.trials[TRIAL_ENT].params
n_niv = pa['n_niveles']
ARQ = dict(n_niveles=n_niv, n_filtros_base=pa['n_filtros_base'],
           kernel_sizes=[pa[f'kernel_nivel_{i}'] for i in range(n_niv)],
           dropout_ps=[pa[f'dropout_nivel_{i}'] for i in range(n_niv)])
lr, alpha = pe['lr'], pe['alpha']
epocas, batch_size = pe['epocas'], pe['batch_size']
print('Arquitectura (Fase 1, ensayo 106):', ARQ)
print(f'Entrenamiento (Fase 2, ensayo 39): lr={lr}, alpha={alpha}, '
      f'épocas={epocas}, batch={batch_size}')
print('Atributos guardados del ensayo 39:',
      est_ent.trials[TRIAL_ENT].user_attrs)

# ---- Igual que objective_entrenamiento, en el mismo orden ----------
torch.cuda.empty_cache()
generator = torch.Generator()
generator.manual_seed(SEED)
train_loader_r = DataLoader(train_dataset, batch_size=batch_size,
                            shuffle=True, generator=generator,
                            num_workers=0, pin_memory=True)
val_loader_r = DataLoader(val_dataset, batch_size=BATCH_VAL,
                          shuffle=False, num_workers=0, pin_memory=True)
torch.manual_seed(SEED + TRIAL_ENT)
np.random.seed(SEED + TRIAL_ENT)
modelo_r = UNet_WRF(in_channels=3, **ARQ).to(device)
print('Parámetros:', sum(p.numel() for p in modelo_r.parameters()),
      '(tiene que dar 9.555.457)')
criterion_r = FACL(total_steps=epocas * len(train_loader_r),
                   alpha=alpha).to(device)
optimizer_r = optim.Adam(modelo_r.parameters(), lr=lr)
scheduler_r = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_r, T_max=epocas, eta_min=1e-7)

hist = dict(epoca=[], L_val=[], FAL=[], FCL=[], prob_fcl=[],
            frac_pasos_fcl=[], lr=[], segundos=[])
inicio = 0
if os.path.exists(CKPT):                      # reanudar
    ck = torch.load(CKPT, map_location=device, weights_only=False)
    modelo_r.load_state_dict(ck['modelo'])
    optimizer_r.load_state_dict(ck['optimizer'])
    scheduler_r.load_state_dict(ck['scheduler'])
    criterion_r.step = ck['criterion_step']
    torch.set_rng_state(ck['rng_torch'])
    torch.cuda.set_rng_state_all(ck['rng_cuda'])
    np.random.set_state(ck['rng_numpy'])
    generator.set_state(ck['rng_loader'])
    hist, inicio = ck['hist'], ck['epoca']
    print(f'Reanudando desde la época {inicio + 1}')

for epoch in range(inicio, EPOCA_FINAL):
    t0 = time.time()
    lr_epoca = optimizer_r.param_groups[0]['lr']
    modelo_r.train()
    n_fcl = n_tot = 0
    for batch_X, batch_y in tqdm(train_loader_r, leave=False,
                                 desc=f'Época {epoch + 1}/{epocas}'):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer_r.zero_grad()
        loss = criterion_r(modelo_r(batch_X), batch_y, mode='train')
        loss.backward()
        optimizer_r.step()
        n_tot += 1
    # Fracción esperada de pasos con FCL en la época (P(t) promedio)
    paso_fin = criterion_r.step
    frac = np.mean([criterion_r.prob_fcl(t)
                    for t in range(paso_fin - n_tot, paso_fin)])

    modelo_r.eval()
    criterion_r.val_reset()
    with torch.no_grad():
        for batch_X, batch_y in val_loader_r:
            criterion_r.val_acumular(modelo_r(batch_X.to(device)),
                                     batch_y.to(device))
    lval = criterion_r.val_resultado()
    scheduler_r.step()

    for k, v in (('epoca', epoch + 1), ('L_val', lval),
                 ('FAL', criterion_r.val_fal),
                 ('FCL', criterion_r.val_fcl),
                 ('prob_fcl', criterion_r.prob_fcl()),
                 ('frac_pasos_fcl', float(frac)), ('lr', lr_epoca),
                 ('segundos', time.time() - t0)):
        hist[k].append(float(v))
    print(f'Época {epoch + 1}/{epocas} | L_val={lval:.6f} | '
          f'FAL={criterion_r.val_fal:.6f} FCL={criterion_r.val_fcl:.6f}'
          f' | P(FCL)={criterion_r.prob_fcl():.3f} | '
          f'{time.time() - t0:.0f}s')

    with open(SALIDA, 'w', encoding='utf-8') as f:
        json.dump(dict(arquitectura=ARQ, lr=lr, alpha=alpha,
                       epocas=epocas, batch_size=batch_size,
                       hist=hist), f, indent=1)
    torch.save(dict(modelo=modelo_r.state_dict(),
                    optimizer=optimizer_r.state_dict(),
                    scheduler=scheduler_r.state_dict(),
                    criterion_step=criterion_r.step,
                    rng_torch=torch.get_rng_state(),
                    rng_cuda=torch.cuda.get_rng_state_all(),
                    rng_numpy=np.random.get_state(),
                    rng_loader=generator.get_state(),
                    hist=hist, epoca=epoch + 1), CKPT)

print('\nControl: época 8  L_val =', round(hist['L_val'][-1], 6),
      '(en la reproducción de las curvas: 0,046700; tiene que ser',
      'el mínimo de L_val)')
print('         mínimo de L_val hasta ahora en la época',
      hist['epoca'][int(np.argmin(hist['L_val']))], '(esperada: 8)')
print(f'Guardado {SALIDA} y {CKPT} (pesos de la época 8)')

# =====================================================================
# PARTE 2: estimaciones del modelo de la época 8 sobre la validación
# Hace falta resultados_validacion.npz (el que ya generó
# celda_resultados_validacion.py) en la carpeta actual. Agrega la
# clave `pred_ext_epoca8` y guarda resultados_validacion_epoca8.npz.
# =====================================================================
DIR_EXT = '../dataset'          # conjunto extendido, 3 canales
RUTA_NPZ = 'resultados_validacion.npz'
RUTA_SALIDA = 'resultados_validacion_epoca8.npz'
LOTE, TAM = 16, 297

ck8 = torch.load(CKPT, map_location=device, weights_only=False)
assert ck8['epoca'] == EPOCA_FINAL, ck8['epoca']
m8 = UNet_WRF(in_channels=3, **ARQ).to(device)
m8.load_state_dict(ck8['modelo'])
m8.eval()

previo = dict(np.load(RUTA_NPZ))
nombres = [str(n) for n in previo['archivos']]
ds_e = TormentaDataset(split_dir=f'{DIR_EXT}/val',
                       stats_path=f'{DIR_EXT}/stats.npz')
idx_e = {f.name: i for i, f in enumerate(ds_e.files)}
assert all(n in idx_e for n in nombres), 'faltan muestras'

out = np.zeros((len(nombres), TAM, TAM), dtype=np.uint8)
with torch.no_grad():
    for a in tqdm(range(0, len(nombres), LOTE), desc='época 8'):
        lote = nombres[a:a + LOTE]
        X = torch.stack([ds_e[idx_e[n]][0] for n in lote]).to(device)
        p = m8(X).cpu().squeeze(1).clamp(0, 1)
        for k in range(len(lote)):
            v = np.round(ds_e.desnormalizar_y(p[k]).numpy())
            assert v.min() >= 0 and v.max() <= 255
            out[a + k] = v.astype(np.uint8)
previo['pred_ext_epoca8'] = out

real = previo['y_real']
act = lambda M: M.reshape(len(M), -1).sum(1, dtype=np.int64)
print('Cociente estimada/simulada (todo el conjunto):')
for k in ('pred_ext_ultima', 'pred_ext_epoca8', 'pred_comb_mejor'):
    print(f'  {k:18s}: {act(previo[k]).sum() / act(real).sum():.2f}')
np.savez_compressed(RUTA_SALIDA, **previo)
print(f'Guardado {RUTA_SALIDA} '
      f'({os.path.getsize(RUTA_SALIDA) / 1e6:.1f} MB). '
      'Mandame ese archivo y esta salida.')

In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================
# CELDA PARA Entrenamiento_Ext_Clip_U-Net_FACL.ipynb  (se puede correr
# en el JupyterHub de Serafín o en Clementina; hace falta una GPU)
#
# Qué hace: obtiene los pesos de la Ext Clip U-Net FACL en su "mejor
# época según L_val" (la 8), que la reproducción de las curvas no
# guardó, para mostrar en Resultados cómo cambia la serie temporal de
# validación según qué pérdida de validación elige la época.
#   PARTE 1: reentrena el ensayo 39 de la Fase 2 con exactamente lo
#            mismo que celda_curvas_fal_fcl_trial39.py (semillas,
#            hiperparámetros leídos de Optuna, arquitectura del
#            ensayo 106), pero SOLO LAS PRIMERAS 8 ÉPOCAS, con el
#            cronograma de las 140 (T_max, FACL y probabilidad de
#            usar la FCL), y guarda los pesos de la época 8.
#            En Clementina dura ~1 h 10 min (8 épocas de ~8,3 min);
#            en el JupyterHub tardará más (la Ext U-Net Comb
#            V2.1, de 8 millones de parámetros, tardó ~25 min por
#            época ahí), así que contá con unas 3 o 4 horas.
#   PARTE 2: estima con ese modelo las 2.271 muestras de validación y
#            guarda resultados_validacion_epoca8.npz (el archivo
#            anterior más la clave `pred_ext_epoca8`).
#
# Antes de correrla, ejecutá las celdas de: Dependencias, semillas
# (SEED = 42), UNet_WRF, FACL, DATASETS (incluye TormentaDataset) e
# `import optuna`. Copiá resultados_validacion.npz a la carpeta
# desde la que corrés (o cambiá RUTA_NPZ). Si se corta, volvé a
# correrla: sigue desde la última época completa.
#
# Salidas: ../modelos/etapa_B_Ext_Clip_U-Net_FACL/
#              B_trial0039_repro_epoca8.pth
#          resultados_validacion_epoca8.npz   <- mandame este
# =====================================================================
import glob
import json
import os
import time

TRIAL_ARQ, TRIAL_ENT = 106, 39
BATCH_VAL = 16
EPOCA_FINAL = 8      # se entrena con el cronograma de 140 épocas
SALIDA = f'../estudios/trial{TRIAL_ENT:04d}_hasta_epoca8.json'
CKPT = (f'../modelos/etapa_B_Ext_Clip_U-Net_FACL/'
        f'B_trial{TRIAL_ENT:04d}_repro_epoca8.pth')


def buscar_estudio(patron):
    """Primer estudio de Optuna cuyo .db coincide con el patrón."""
    for db in sorted(glob.glob(f'../estudios/{patron}')):
        storage = f'sqlite:///{db}'
        for resumen in optuna.get_all_study_summaries(storage):
            print(f'  {os.path.basename(db)} -> {resumen.study_name}')
            return optuna.load_study(study_name=resumen.study_name,
                                     storage=storage)
    raise FileNotFoundError(f'No encontré ../estudios/{patron}')


print('Estudios encontrados:')
est_arq = buscar_estudio('optuna_arquitectura_*Ext Clip U-Net FACL*.db')
est_ent = buscar_estudio('optuna_entrenamiento_*Ext Clip U-Net FACL*.db')
pa = est_arq.trials[TRIAL_ARQ].params
pe = est_ent.trials[TRIAL_ENT].params
n_niv = pa['n_niveles']
ARQ = dict(n_niveles=n_niv, n_filtros_base=pa['n_filtros_base'],
           kernel_sizes=[pa[f'kernel_nivel_{i}'] for i in range(n_niv)],
           dropout_ps=[pa[f'dropout_nivel_{i}'] for i in range(n_niv)])
lr, alpha = pe['lr'], pe['alpha']
epocas, batch_size = pe['epocas'], pe['batch_size']
print('Arquitectura (Fase 1, ensayo 106):', ARQ)
print(f'Entrenamiento (Fase 2, ensayo 39): lr={lr}, alpha={alpha}, '
      f'épocas={epocas}, batch={batch_size}')
print('Atributos guardados del ensayo 39:',
      est_ent.trials[TRIAL_ENT].user_attrs)

# ---- Igual que objective_entrenamiento, en el mismo orden ----------
torch.cuda.empty_cache()
generator = torch.Generator()
generator.manual_seed(SEED)
train_loader_r = DataLoader(train_dataset, batch_size=batch_size,
                            shuffle=True, generator=generator,
                            num_workers=0, pin_memory=True)
val_loader_r = DataLoader(val_dataset, batch_size=BATCH_VAL,
                          shuffle=False, num_workers=0, pin_memory=True)
torch.manual_seed(SEED + TRIAL_ENT)
np.random.seed(SEED + TRIAL_ENT)
modelo_r = UNet_WRF(in_channels=3, **ARQ).to(device)
print('Parámetros:', sum(p.numel() for p in modelo_r.parameters()),
      '(tiene que dar 9.555.457)')
criterion_r = FACL(total_steps=epocas * len(train_loader_r),
                   alpha=alpha).to(device)
optimizer_r = optim.Adam(modelo_r.parameters(), lr=lr)
scheduler_r = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer_r, T_max=epocas, eta_min=1e-7)

hist = dict(epoca=[], L_val=[], FAL=[], FCL=[], prob_fcl=[],
            frac_pasos_fcl=[], lr=[], segundos=[])
inicio = 0
if os.path.exists(CKPT):                      # reanudar
    ck = torch.load(CKPT, map_location=device, weights_only=False)
    modelo_r.load_state_dict(ck['modelo'])
    optimizer_r.load_state_dict(ck['optimizer'])
    scheduler_r.load_state_dict(ck['scheduler'])
    criterion_r.step = ck['criterion_step']
    # map_location=device mueve los estados del RNG a la GPU, y
    # set_rng_state exige ByteTensor en la CPU: se los devuelve ahí.
    torch.set_rng_state(ck['rng_torch'].cpu())
    torch.cuda.set_rng_state_all([t.cpu() for t in ck['rng_cuda']])
    np.random.set_state(ck['rng_numpy'])
    generator.set_state(ck['rng_loader'].cpu())
    hist, inicio = ck['hist'], ck['epoca']
    print(f'Reanudando desde la época {inicio + 1}')

for epoch in range(inicio, EPOCA_FINAL):
    t0 = time.time()
    lr_epoca = optimizer_r.param_groups[0]['lr']
    modelo_r.train()
    n_fcl = n_tot = 0
    for batch_X, batch_y in tqdm(train_loader_r, leave=False,
                                 desc=f'Época {epoch + 1}/{epocas}'):
        batch_X, batch_y = batch_X.to(device), batch_y.to(device)
        optimizer_r.zero_grad()
        loss = criterion_r(modelo_r(batch_X), batch_y, mode='train')
        loss.backward()
        optimizer_r.step()
        n_tot += 1
    # Fracción esperada de pasos con FCL en la época (P(t) promedio)
    paso_fin = criterion_r.step
    frac = np.mean([criterion_r.prob_fcl(t)
                    for t in range(paso_fin - n_tot, paso_fin)])

    modelo_r.eval()
    criterion_r.val_reset()
    with torch.no_grad():
        for batch_X, batch_y in val_loader_r:
            criterion_r.val_acumular(modelo_r(batch_X.to(device)),
                                     batch_y.to(device))
    lval = criterion_r.val_resultado()
    scheduler_r.step()

    for k, v in (('epoca', epoch + 1), ('L_val', lval),
                 ('FAL', criterion_r.val_fal),
                 ('FCL', criterion_r.val_fcl),
                 ('prob_fcl', criterion_r.prob_fcl()),
                 ('frac_pasos_fcl', float(frac)), ('lr', lr_epoca),
                 ('segundos', time.time() - t0)):
        hist[k].append(float(v))
    print(f'Época {epoch + 1}/{epocas} | L_val={lval:.6f} | '
          f'FAL={criterion_r.val_fal:.6f} FCL={criterion_r.val_fcl:.6f}'
          f' | P(FCL)={criterion_r.prob_fcl():.3f} | '
          f'{time.time() - t0:.0f}s')

    with open(SALIDA, 'w', encoding='utf-8') as f:
        json.dump(dict(arquitectura=ARQ, lr=lr, alpha=alpha,
                       epocas=epocas, batch_size=batch_size,
                       hist=hist), f, indent=1)
    torch.save(dict(modelo=modelo_r.state_dict(),
                    optimizer=optimizer_r.state_dict(),
                    scheduler=scheduler_r.state_dict(),
                    criterion_step=criterion_r.step,
                    rng_torch=torch.get_rng_state(),
                    rng_cuda=torch.cuda.get_rng_state_all(),
                    rng_numpy=np.random.get_state(),
                    rng_loader=generator.get_state(),
                    hist=hist, epoca=epoch + 1), CKPT)

print('\nControl: época 8  L_val =', round(hist['L_val'][-1], 6),
      '(en la reproducción de las curvas: 0,046700; tiene que ser',
      'el mínimo de L_val)')
print('         mínimo de L_val hasta ahora en la época',
      hist['epoca'][int(np.argmin(hist['L_val']))], '(esperada: 8)')
print(f'Guardado {SALIDA} y {CKPT} (pesos de la época 8)')

# =====================================================================
# PARTE 2: estimaciones del modelo de la época 8 sobre la validación
# Hace falta resultados_validacion.npz (el que ya generó
# celda_resultados_validacion.py) en la carpeta actual. Agrega la
# clave `pred_ext_epoca8` y guarda resultados_validacion_epoca8.npz.
# =====================================================================
DIR_EXT = '../dataset'          # conjunto extendido, 3 canales
RUTA_NPZ = 'resultados_validacion.npz'
RUTA_SALIDA = 'resultados_validacion_epoca8.npz'
LOTE, TAM = 16, 297

ck8 = torch.load(CKPT, map_location=device, weights_only=False)
assert ck8['epoca'] == EPOCA_FINAL, ck8['epoca']
m8 = UNet_WRF(in_channels=3, **ARQ).to(device)
m8.load_state_dict(ck8['modelo'])
m8.eval()

previo = dict(np.load(RUTA_NPZ))
nombres = [str(n) for n in previo['archivos']]
ds_e = TormentaDataset(split_dir=f'{DIR_EXT}/val',
                       stats_path=f'{DIR_EXT}/stats.npz')
idx_e = {f.name: i for i, f in enumerate(ds_e.files)}
assert all(n in idx_e for n in nombres), 'faltan muestras'

out = np.zeros((len(nombres), TAM, TAM), dtype=np.uint8)
with torch.no_grad():
    for a in tqdm(range(0, len(nombres), LOTE), desc='época 8'):
        lote = nombres[a:a + LOTE]
        X = torch.stack([ds_e[idx_e[n]][0] for n in lote]).to(device)
        p = m8(X).cpu().squeeze(1).clamp(0, 1)
        for k in range(len(lote)):
            v = np.round(ds_e.desnormalizar_y(p[k]).numpy())
            assert v.min() >= 0 and v.max() <= 255
            out[a + k] = v.astype(np.uint8)
previo['pred_ext_epoca8'] = out

real = previo['y_real']
act = lambda M: M.reshape(len(M), -1).sum(1, dtype=np.int64)
print('Cociente estimada/simulada (todo el conjunto):')
for k in ('pred_ext_ultima', 'pred_ext_epoca8', 'pred_comb_mejor'):
    print(f'  {k:18s}: {act(previo[k]).sum() / act(real).sum():.2f}')
np.savez_compressed(RUTA_SALIDA, **previo)
print(f'Guardado {RUTA_SALIDA} '
      f'({os.path.getsize(RUTA_SALIDA) / 1e6:.1f} MB). '
      'Mandame ese archivo y esta salida.')

In [ ]:
# -*- coding: utf-8 -*-
# =====================================================================
# CELDA PARA CUALQUIER NOTEBOOK (sin GPU) - versión rápida
#
# La versión anterior le hacía un stat a cada .npz; en la carpeta
# temporal (disco de red, caché fría) eso tardó más de 2 horas.
# Esta cuenta los archivos con una sola lectura del directorio
# (os.listdir, sin stat) y mide el tamaño en disco de una MUESTRA de
# archivos por carpeta, mostrando el avance y el tiempo.
#   - memoria: exacta (nº de archivos x forma de X e y x 4 bytes);
#   - disco:   exacto si todos los archivos de la carpeta se midieron
#              (MUESTRA = None); si no, estimado = media x cantidad.
# Antes de correrla: interrumpí la celda anterior (Kernel > Interrupt).
# Salida: ../estudios/tamano_datasets.txt  <- mandame este
# =====================================================================
import os
import random
import time
from pathlib import Path

import numpy as np

CARPETAS = {'Ext (3 canales)': '../dataset',
            'Temp (9 canales)': '../dataset_temporal'}
MUESTRA = 300          # archivos medidos por carpeta (None = todos)
random.seed(0)
lineas = []


def anotar(t):
    print(t, flush=True)
    lineas.append(t)


for nombre, base in CARPETAS.items():
    anotar(f'== {nombre}: {base}')
    tot_disco = tot_mem = tot_n = 0
    exacto = True
    for split in ('train', 'val'):
        t0 = time.time()
        carpeta = Path(base, split)
        nombres = sorted(n for n in os.listdir(carpeta)
                         if n.endswith('.npz'))
        print(f'  {split}: {len(nombres)} archivos listados en '
              f'{time.time() - t0:.1f} s', flush=True)
        elegidos = (nombres if MUESTRA is None or
                    len(nombres) <= MUESTRA
                    else random.sample(nombres, MUESTRA))
        t0 = time.time()
        tam = []
        for k, n in enumerate(elegidos):
            tam.append(os.stat(carpeta / n).st_size)
            if k % 50 == 0:
                print(f'    stat {k}/{len(elegidos)} '
                      f'({time.time() - t0:.1f} s)', flush=True)
        medido_todo = len(elegidos) == len(nombres)
        exacto &= medido_todo
        disco = (sum(tam) if medido_todo
                 else np.mean(tam) * len(nombres))
        with np.load(carpeta / nombres[0]) as d:
            fx, fy = tuple(d['X'].shape), tuple(d['y'].shape)
            mem = len(nombres) * (d['X'].size + d['y'].size) * 4
        anotar(f"  {split:5s}: {len(nombres):6d} muestras | X {fx}, "
               f"y {fy} | disco {disco / 1e6:9.1f} MB"
               f"{'' if medido_todo else ' (estimado)'} | memoria "
               f"float32 {mem / 1e6:9.1f} MB")
        tot_disco += disco
        tot_mem += mem
        tot_n += len(nombres)
    anotar(f'  total: {tot_n} muestras | disco {tot_disco / 1e6:.1f} '
           f"MB ({tot_disco / 2**30:.2f} GiB)"
           f"{'' if exacto else ' (estimado)'} | memoria float32 "
           f'{tot_mem / 1e6:.1f} MB ({tot_mem / 2**30:.2f} GiB)')
Path('../estudios').mkdir(exist_ok=True)
Path('../estudios/tamano_datasets.txt').write_text('\n'.join(lineas),
                                                   encoding='utf-8')
print('Guardado ../estudios/tamano_datasets.txt')